#### PIP

In [165]:
%pip install -q nltk
%pip install -q spacy
%pip install -q gensim


[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports and Downloads

In [1]:
import nltk

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('universal_tagset', quiet=True)

True

In [76]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import spacy
from spacy.tokens import Doc

# Load the spaCy model globally (en_core_web_sm is lightweight and efficient)
nlp = spacy.load("en_core_web_sm")

In [127]:
import numpy as np
import gensim.downloader as api

print("Global load: Downloading/Loading Word2Vec model...")
GLOBAL_W2V_MODEL = api.load('word2vec-google-news-300')
print("Global load: Word2Vec model ready.")

Global load: Downloading/Loading Word2Vec model...
Global load: Word2Vec model ready.


---

#### Set Notebook Seed

In [105]:
SEED=142

In [ ]:
import torch
import random
import numpy as np

def set_seed(seed: int = 142):
    """Locks all random number generators for exact reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed}")

In [111]:
set_seed(SEED)

Global seed set to 142


## Task 1

#### PropagandaFeaturePipeline (Class)

In [166]:
import re
import csv
from collections import Counter
import torch
import spacy
from spacy.tokens import Doc
from nltk.tag.perceptron import PerceptronTagger
from nltk.tag import map_tag

class PropagandaFeaturePipeline:
    """
    Encapsulates state (vocabularies, tagsets) while maintaining a pure 
    functional approach to row-by-row string processing and vectorization.
    """
    def __init__(self, spacy_model="en_core_web_sm", exclude_non_propaganda=True):

        self.LABELS = [
            'name_calling,labeling', 'repetition', 'causal_oversimplification', 
            'doubt', 'loaded_language', 'appeal_to_fear_prejudice', 
            'flag_waving', 'exaggeration,minimisation', 'not_propaganda'
        ]

        # For Task 1
        if exclude_non_propaganda:
            self.LABELS.remove('not_propaganda')
        
        self.UNIVERSAL_TAGSET = ["ADJ","ADP","ADV","CONJ","DET","NOUN","NUM","PRT","PRON","VERB",".","X"]
        
        # Custom (Shortened) NER Tagset
        self.NER_TAG = ['PERSON','ORG','GPE','DATE','NORP','CARDINAL','ORDINAL','TIME','LOC', 'O']
        
        # Top N most frequent words
        self.CUSTOM_STOPWORDS = ["the" , ",", "to", "of", "and", "in", "a", "that"]

        self.word_to_index = {}     # bow vector indicies
        self.word_to_index_silver = {}  # bow vector indicies using synthetic data
        self.hapax_words_list = []
        self.hapax_words_list_silver = []
        
        self.pos_to_index = self._build_tag_index(self.UNIVERSAL_TAGSET)    # POS tagset vector indicies
        self.ner_to_index = self._build_tag_index(["MISC"] + self.NER_TAG)  # NER tagset vector indicies

        self.nlp = spacy.load(spacy_model)  # taggers
        self.tagger = PerceptronTagger()

        self.w2v_model = None

    # ==========================================
    # INTERNAL BUILDER METHODS
    # ==========================================

    def _build_tag_index(self, tagset: list[str]) -> dict[str, int]:
        """Creates a mapping of tags to index positions."""
        return {tag: i for i, tag in enumerate(tagset) if tag not in ["O", "__BOUNDARY__"]}

    
    def build_vocabularies(self, gold_path: str, silver_path: str, full_context: str = True):
        """
        Parses the datasets to populate the class-level vocabulary matrices.
        This replaces the global counter loops from the notebook.
        """
        global_vocab = Counter()
        
        # Build Gold Vocab
        with open(gold_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in tsv_reader:
                label, text = self.process_row(row)
                if label not in self.LABELS: continue   # skip `not_propaganda` 
                tokens = self.tokenize_whole_words(text)

        

                if full_context:
                    global_vocab.update(tokens)
                else:    
                    in_snippet = False
                    for token in tokens:
                        if token == "<BOS>": in_snippet = True; continue
                        if token == "<EOS>": in_snippet = False; continue
                        if in_snippet:
                            global_vocab[token] += 1

        global_vocab_silver = global_vocab.copy()
                
        # Build Silver Vocab
        with open(silver_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in tsv_reader:
                label, text = self.process_row(row)
                if label not in self.LABELS: continue
                tokens = self.tokenize_whole_words(text) 
                
                in_snippet = False  # only draw silver counts from synthetic snippet
                for token in tokens:
                    if token == "<BOS>": in_snippet = True; continue
                    if token == "<EOS>": in_snippet = False; continue
                    if in_snippet and token in global_vocab:
                        global_vocab_silver[token] += 1

        # States
        self.hapax_words_list = [word for word, count in global_vocab.items() if count == 1]
        self.hapax_words_list_silver = [word for word, count in global_vocab_silver.items() if count == 1]
        
        gold_list = ["__UNK__"] + [word for word, count in global_vocab.items() if count > 1]
        silver_list = ["__UNK__"] + [word for word, count in global_vocab_silver.items() if count > 1]
        
        self.word_to_index = {w: i for i, w in enumerate(w for w in gold_list if w not in self.CUSTOM_STOPWORDS + ["<EOS>","<BOS>"])}
        self.word_to_index_silver = {w: i for i, w in enumerate(w for w in silver_list if w not in self.CUSTOM_STOPWORDS + ["<EOS>","<BOS>"])}
        print("Vocabulary State Successfully Initialized.")


    # ==========================================
    # FUNCTIONAL TEXT PROCESSING ZONE
    # ==========================================
    
    def universal_cleaning(self, raw_text: str) -> str:
        """Cleaning directly on raw STRING format"""
        text = raw_text.strip() # clear leading/trailing whitespace
        text = text.replace("\\'", "'").replace('\\"', '"') # strip out python escape backslashes
        text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'") # standardize quotes to flat quotes
        text = re.sub(r"(?<=\w)'(?=\w)|(?<=[sS])'", '', text) # collapse intra-word apostrophes: won't -> wont, lukes' -> lukes
        text = re.sub(r'[\\/\[\]*|@\ \-.:$#+=]', ' ', text) # remove artifacts: \ / [ ] * | @ space - . : $ # + =
        text = text.replace("<BOS>", " <BOS> ").replace("<EOS>", " <EOS> ") # ensure space around bound tags
        return " ".join(text.split())

    def process_row(self, row: dict) -> tuple[str, str]:
        """Process raw row directly from csv"""
        return row['label'], self.universal_cleaning(row['tagged_in_context'])

    def tokenize_whole_words(self, text: str) -> list[str]:
        """Turn string text into whole-word tokens using regex parser"""
        localized_text = re.sub(r'\b\d+(?:,\d+)*\b', 'num', text)
        pattern = r"<BOS>|<EOS>|(?:[a-zA-Z]\.)+|[a-zA-Z0-9]+(?:[-']?[a-zA-Z0-9]+)*|[^\w\s]"
        raw_tokens = re.findall(pattern, localized_text)
        return [t if t in ["<BOS>", "<EOS>"] else t.lower() for t in raw_tokens]

    def tag_pos_pipeline(self, text: str) -> list[str]:
        """Turn string text into pos tokens using NLTK Perceptron"""
        tokens = self.tokenize_whole_words(text)
        raw_tags = self.tagger.tag(tokens) # nltk PerceptronTagger
        return [
            ("__BOUNDARY__") if t == "<BOS>" or t == "<EOS>" else
            ("NUM") if t.lower() == "num" else # capture num rule from string formatting
            (".") if t in ['"', "'", '`'] else # override mapping
            (map_tag('en-ptb', 'universal', tag)) # map from perceptron native pentree to universal tags
            for t, tag in raw_tags
        ]

    def tag_ner_pipeline(self, text: str) -> list[str]:
        """Turn string text into NER tokens using Spacy"""
        
        allowed = set(self.NER_TAG)
        tokens = self.tokenize_whole_words(text)

        doc = Doc(self.nlp.vocab, words=tokens)
        for name, proc in self.nlp.pipeline: doc = proc(doc)
            
        ner_tags = []
        for token in doc:
            if token.text in ["<BOS>", "<EOS>"]: ner_tags.append("__BOUNDARY__")
            elif token.ent_type_:
                # Custom NER list excludes low count tags, route these into MISC category
                ner_tags.append(f"{token.ent_type_}" if token.ent_type_ in allowed else "MISC")
            else: ner_tags.append("O")
        return ner_tags


    # ==========================================
    # VECTORIZATION ZONE
    # ==========================================

    def string_to_word2vec_vector(self, string: str, use_silver: bool = False) -> list[float]:
        """
        Turn text string into a 300D Word2Vec mean-pooled vector.
        Only processes words that exist in our learned vocabularies.
        """
        if self.w2v_model is None:
            self.load_word2vec()

        active_vocab = self.word_to_index_silver if use_silver else self.word_to_index
        tokenized = self.tokenize_whole_words(string)
        
        vectors = []
        unk_count = 0
        for token in tokenized:
            if token in ["<EOS>", "<BOS>"] or token in self.CUSTOM_STOPWORDS: continue  
            if token not in active_vocab: 
                unk_count += 1 
                continue 
                
            if token in self.w2v_model:
                vectors.append(self.w2v_model[token])
            elif token.capitalize() in self.w2v_model: # fallback check cap version
                vectors.append(self.w2v_model[token.capitalize()])
                # will not match names, punct or obsurce words
                
        if len(vectors) > 0:
            mean_vector = np.mean(vectors, axis=0).tolist() # Mean Pooling
        else:
            mean_vector = [0.0] * 300 # no match fallback
            print("empty w2v vector")
        
        unk_count = unk_count / len(tokenized) if len(tokenized) > 0 else 0.0 # normalize
            
        return mean_vector, unk_count


    def string_to_bow_vector(self, string: str, use_silver: bool = False) -> list[int]:
        """Turn text string into a vocab bow python vector"""
        active_vocab = self.word_to_index_silver if use_silver else self.word_to_index
        tokenized = self.tokenize_whole_words(string)
        sequence_vector = [0] * len(active_vocab)

        for token in tokenized:
            # avoiding counting boundaries and stopwords
            if token in ["<EOS>", "<BOS>"] or token in self.CUSTOM_STOPWORDS: continue
            
            # populate sparse vector + unk index
            idx = active_vocab.get(token, active_vocab["__UNK__"])
            sequence_vector[idx] += 1

        return sequence_vector


    def tagset_to_vector(self, string: str, tag_type: str) -> list[int]:
        """Turn text string into a tagset bow python vector"""
        if tag_type == "POS":
            tag_list = self.tag_pos_pipeline(string) 
            active_index = self.pos_to_index
        elif tag_type == "NER":
            tag_list = self.tag_ner_pipeline(string)
            active_index = self.ner_to_index

        sequence_vector = [0] * len(active_index)
        for tag in tag_list:
            if tag in ["__BOUNDARY__", "O"]: continue
            sequence_vector[active_index[tag]] += 1
        return sequence_vector


    def string_to_input_vector(
            self, 
            string_text: str, 
            use_silver: bool = False,
            feature_type: str = "bow"
            ) -> torch.Tensor:
        """
        Routes text string to the correct token vectorization method,
        generates POS/NER vectors, and concatenates them for MLP input.
        """

        # Generate token vectors
        if feature_type == "word2vec":
            text_vector, unk_count = self.string_to_word2vec_vector(string_text, use_silver)
            t_vocab = torch.tensor(text_vector + [unk_count], dtype=torch.float32)
            
        elif feature_type == "bow":
            text_vector = self.string_to_bow_vector(string_text, use_silver)
            t_vocab = torch.tensor(text_vector, dtype=torch.float32)

        # Generate tag vectors
        pos_vector = self.tagset_to_vector(string_text, "POS")
        ner_vector = self.tagset_to_vector(string_text, "NER")

        # pytorch tensor normalisation
        t_pos = torch.tensor(pos_vector, dtype=torch.float32)
        t_ner = torch.tensor(ner_vector, dtype=torch.float32)

        if feature_type == "word2vec":
            # convert tagset counts to distrubution to match w2v magnitutde
            if t_pos.sum() > 0: t_pos = t_pos / t_pos.sum()
            if t_ner.sum() > 0: t_ner = t_ner / t_ner.sum() # converts space into distribtuion

        x_combined = torch.cat([t_vocab, t_pos, t_ner], dim=0)

        return x_combined.unsqueeze(0)
    

    # ==========================================
    # LOADING ZONE
    # ==========================================

    def load_word2vec(self):
        """Pulls the pre-loaded global Word2Vec model."""
        if self.w2v_model is None:
            global GLOBAL_W2V_MODEL
            if 'GLOBAL_W2V_MODEL' in globals() and GLOBAL_W2V_MODEL is not None:
                self.w2v_model = GLOBAL_W2V_MODEL
                print("Word2Vec model successfully linked from global scope.")
            else:
                raise RuntimeError(
                    "GLOBAL_W2V_MODEL is not defined. Please run the global loading cell at the top of the notebook first."
                )

---

##### Example Run:

In [113]:
set_seed(SEED)

pipeline = PropagandaFeaturePipeline()

pipeline.build_vocabularies(
    gold_path='../data/propaganda_train_100.tsv', 
    silver_path='../data/silver_train.tsv', 
    full_context=True
)

print(f"List of corpus labels:             {pipeline.LABELS}")
print(f"Universal POS Tagset:              {pipeline.UNIVERSAL_TAGSET}")
print(f"Custom Simplified NER tagset:      {pipeline.NER_TAG}")
print(f"Custom Stopword List:              {pipeline.CUSTOM_STOPWORDS}")
print(f"Dims of baseline gold sparse vec:  {len(pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(pipeline.ner_to_index)}")

print(f"POS Tagger:                        {pipeline.tagger}")
print(f"NER Tagger:                        {pipeline.nlp}")


Global seed set to 142
Vocabulary State Successfully Initialized.
List of corpus labels:             ['name_calling,labeling', 'repetition', 'causal_oversimplification', 'doubt', 'loaded_language', 'appeal_to_fear_prejudice', 'flag_waving', 'exaggeration,minimisation']
Universal POS Tagset:              ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRT', 'PRON', 'VERB', '.', 'X']
Custom Simplified NER tagset:      ['PERSON', 'ORG', 'GPE', 'DATE', 'NORP', 'CARDINAL', 'ORDINAL', 'TIME', 'LOC', 'O']
Custom Stopword List:              ['the', ',', 'to', 'of', 'and', 'in', 'a', 'that']
Dims of baseline gold sparse vec:  377
Dims of gold + silver sparse vec:  869
Gold Only Singletons:              858
Silver Enriched Singletons:        366
Dims of POS vector:                12
Dimes of NER vector:               10
POS Tagger:                        <nltk.tag.perceptron.PerceptronTagger object at 0x15ab115d0>
NER Tagger:                        <spacy.lang.en.English object at 0x41ea5ae

#### PropagandaTrainer (Class)

In [167]:
import csv
import torch
import torch.nn as nn
import random

class PropagandaTrainer:
    """
    Builds the standardized PyTorch MLP architecture,
    configures optimization, 
    and executes the streaming training loop.
    """
    def __init__(
        self, 
        pipeline, 
        hidden_dim: int = 64, 
        dropout_p: float = 0.3,
        lr: float = 0.0005,
        weight_decay: float = 0.05,
        use_silver: bool = False,    # vocab selector
        feature_type: str = "bow"
    ):
        self.pipeline = pipeline
        self.use_silver = use_silver
        self.feature_type = feature_type
        self.label_to_idx = {label: i for i, label in enumerate(pipeline.LABELS)}
        self.best_val_loss = float('inf')
        
        # Dynamically compute total input dimension from pipeline state
        if feature_type == "word2vec":
            self.pipeline.load_word2vec()
            token_dim = 301
        else:
            active_vocab = pipeline.word_to_index_silver if use_silver else pipeline.word_to_index
            token_dim = len(active_vocab)
        
        # token + tagset vectors
        input_dimension = (
            token_dim + 
            len(pipeline.pos_to_index) + 
            len(pipeline.ner_to_index)
        )
        
        # Build classification head
        self.model = self._build_head(
            input_dim=input_dimension,
            hidden_dim=hidden_dim,
            num_classes=len(pipeline.LABELS),
            dropout_p=dropout_p
        )
        
        # 3. Configure head components
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(), 
            lr=lr, 
            weight_decay=weight_decay
        )

    # ==========================================
    # NETWORK BUILDER
    # ==========================================
    def _build_head(self, input_dim: int, hidden_dim: int, num_classes: int, dropout_p: float) -> nn.Module:
        """
        Constructs the standardized MLP classification head directly.
        Uses LayerNorm instead of BatchNorm1d to ensure stability during batch_size=1 streaming.
        """
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, num_classes)
        )

    # ==========================================
    # STREAMING TRAINING PASS
    # ==========================================
    def run_training_loop(
        self, 
        dataset_path: str, 
        epochs: int = 5, 
        save_path: str = 'propaganda_mlp_weights.pt',
        save_best_only: bool = True
    ) -> list[tuple[int, float, float]]:
        """
        Executes row-by-row streaming training and 10% modulo validation.
        """

        epoch_history = []

        for epoch in range(1, epochs + 1):
            print(f"--- Starting Epoch {epoch} ---")

            running_train_loss, train_samples = 0.0, 0
            running_val_loss, val_samples = 0.0, 0

            with open(dataset_path, mode='r', encoding='utf-8') as file:
                tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
                
                for row_idx, raw_row in enumerate(tsv_reader, start=1):
                    label, text = self.pipeline.process_row(raw_row)

                    # Formatting guardrail
                    if text.count("<BOS>") != 1 or text.count("<EOS>") != 1 or label not in self.label_to_idx:
                        continue

                    # Feature Extraction
                    x_batched = self.pipeline.string_to_input_vector(text, use_silver=self.use_silver, feature_type=self.feature_type)
                    y_target = torch.tensor([self.label_to_idx[label]], dtype=torch.long)
                    
                    # 10% Modulo Split for internal dev validation
                    if row_idx % 10 == 0:
                        self.model.eval() # testing
                        with torch.no_grad():
                            logits = self.model(x_batched)
                            val_loss = self.criterion(logits, y_target)
                            running_val_loss += val_loss.item()
                            val_samples += 1
                    else:
                        self.model.train() # training
                        self.optimizer.zero_grad()
                        logits = self.model(x_batched)
                        loss = self.criterion(logits, y_target)
                        loss.backward()
                        self.optimizer.step()
                        
                        running_train_loss += loss.item()
                        train_samples += 1

            # Epoch reporting
            epoch_train_loss = running_train_loss / train_samples if train_samples > 0 else 0.0
            epoch_val_loss = running_val_loss / val_samples if val_samples > 0 else 0.0
            
            print(f"Epoch {epoch} Results | Avg Train Loss: {epoch_train_loss:.4f} | Avg Dev Loss: {epoch_val_loss:.4f}")

            epoch_history.append((epoch, round(epoch_train_loss, 4), round(epoch_val_loss, 4)))

            if save_best_only:
                # Early stopping / Best Checkpoint Behavior
                if epoch_val_loss < self.best_val_loss:
                    self.best_val_loss = epoch_val_loss
                    torch.save(self.model.state_dict(), save_path)
                    print(f"--> New best validation loss ({self.best_val_loss:.4f}) achieved! Model saved to {save_path}.\n")
                else:
                    print(f"--> No improvement on validation loss. Skipping save.\n")
            else:
                # Fixed N-Epoch Behavior: Always overwrite state dict at every epoch
                torch.save(self.model.state_dict(), save_path)
                print(f"--> Model state updated at Epoch {epoch} and saved to {save_path}.\n")
            
        return epoch_history

    # ==========================================
    # MODEL EVALUATION / INFERENCE PASS
    # ==========================================
    def evaluate(
        self, 
        dataset_path: str, 
        weights_path: str = None, 
        random_guess: bool = False
    ) -> dict:
        """
        Loads saved weights from disk and streams a test/validation file
        to return predictions, targets, and classification metrics.
        """

        # model type router, inc baseline
        if random_guess:
            random.seed(100)
            mode_name = "RANDOM GUESSING BASELINE"
        else:
            if weights_path is None:
                raise ValueError("weights_path must be provided when random_guess=False")
            mode_name = f"MODEL EVALUATION ({weights_path})"
            self.model.load_state_dict(torch.load(weights_path, weights_only=True))
            self.model.eval()

        all_preds = []
        all_targets = []
        idx_to_label = {i: label for label, i in self.label_to_idx.items()}
        num_classes = len(self.label_to_idx)

        # Stream dataset and gather predictions
        with open(dataset_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            
            for row_idx, raw_row in enumerate(tsv_reader, start=1):
                label, text = self.pipeline.process_row(raw_row)

                if text.count("<BOS>") != 1 or text.count("<EOS>") != 1 or label not in self.label_to_idx:
                    continue
                
                y_target = self.label_to_idx[label]

                if random_guess:
                    predicted_idx = random.randint(0, num_classes - 1)
                else:
                    x_batched = self.pipeline.string_to_input_vector(
                        text, 
                        use_silver=self.use_silver,
                        feature_type=self.feature_type
                    )
                    with torch.no_grad():
                        logits = self.model(x_batched)
                        predicted_idx = torch.argmax(logits, dim=1).item()
                
                all_preds.append(predicted_idx)
                all_targets.append(y_target)
    
        # Calculate Performance Metrics
        from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

        # Overall Metrics
        acc = accuracy_score(all_targets, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_targets, all_preds, average='macro', zero_division=0
        )

        print("\n" + "="*50)
        print(f" EVALUATION REPORT: {weights_path}")
        print("="*50)
        print(f" Accuracy:  {acc:.4f}")
        print(f" Macro Precision: {precision:.4f}")
        print(f" Macro Recall:    {recall:.4f}")
        print(f" Macro F1 Score:  {f1:.4f}")
        print("="*50 + "\n")

        # Detailed per-class breakdown
        target_names = [idx_to_label[i] for i in sorted(idx_to_label.keys())]
        print(classification_report(all_targets, all_preds, target_names=target_names, zero_division=0))

        return {
            "accuracy": acc,
            "macro_f1": f1,
            "predictions": [idx_to_label[p] for p in all_preds],
            "targets": [idx_to_label[t] for t in all_targets]
        }

---

##### Example run:

In [115]:
set_seed(SEED)

# 1. Initialize and build feature pipeline state
test_pipeline = PropagandaFeaturePipeline()
test_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv'
)

# 2. Instantiate trainer
test_trainer = PropagandaTrainer(
    pipeline=test_pipeline,
    hidden_dim=64,
    dropout_p=0.3,
    lr=0.0005,
    weight_decay=0.05,
    use_silver=False,
    feature_type="bow"
)

# 3. Launch dynamic training pass
test_trainer.run_training_loop(
    dataset_path='../data/propaganda_train_100.tsv',
    epochs=3,
    save_path='test_propaganda_mlp_weights.pt'
)

Global seed set to 142
Vocabulary State Successfully Initialized.
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.2213 | Avg Dev Loss: 2.1621
--> New best validation loss (2.1621) achieved! Model saved to test_propaganda_mlp_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.6120 | Avg Dev Loss: 2.1611
--> New best validation loss (2.1611) achieved! Model saved to test_propaganda_mlp_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.9761 | Avg Dev Loss: 2.2974
--> No improvement on validation loss. Skipping save.



[(1, 2.2213, 2.1621), (2, 1.612, 2.1611), (3, 0.9761, 2.2974)]

---

### Evaluation

1. [Random Guessing Baseline]()
---
1. [BoW: Full Context, Gold Only](#bow-baseline-full-context-gold-only)
2. [BoW: Full Context, Silver Enriched]()
3. [BoW: Snippet, Gold Only]()
4. [BoW: Snippet, Silver Enriched]()
---
1. [W2V: Full Context, Gold Only]()
2. [W2V: Full Context, Silver Enriched]()
3. [W2V: Snippet, Gold Only]()
4. [W2V: Snippet, Silver Enriched]()
---

#### Random Guessing Baseline

In [119]:
set_seed(SEED)

# Instantiate trainer shell (reuses pipeline setup)
eval_trainer = PropagandaTrainer(pipeline=pipeline)

# 1. Random Guessing Baseline Evaluation
random_results = eval_trainer.evaluate(
    dataset_path='../data/propaganda_val.tsv',
    random_guess=True
)

Global seed set to 142

 EVALUATION REPORT: None
 Accuracy:  0.1392
 Macro Precision: 0.1404
 Macro Recall:    0.1388
 Macro F1 Score:  0.1385

                           precision    recall  f1-score   support

    name_calling,labeling       0.11      0.15      0.13        34
               repetition       0.14      0.15      0.15        40
causal_oversimplification       0.15      0.17      0.16        35
                    doubt       0.15      0.14      0.14        43
          loaded_language       0.14      0.13      0.14        39
 appeal_to_fear_prejudice       0.21      0.16      0.18        43
              flag_waving       0.14      0.11      0.12        45
exaggeration,minimisation       0.08      0.10      0.09        30

                 accuracy                           0.14       309
                macro avg       0.14      0.14      0.14       309
             weighted avg       0.14      0.14      0.14       309



#### BoW Experiments

##### HyperParameter Sweep: BoW Baseline

In [116]:
import itertools

set_seed(SEED)

pipeline = PropagandaFeaturePipeline()
pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=True
)

# 2. Parameter Search Ranges
param_grid = {
    'hidden_dim': [64, 128],
    'lr': [0.001, 0.0005, 0.0001],
    'dropout_p': [0.3, 0.5]
}

# 3. Generate Cartesian combinations
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Tracking metrics
global_best_loss = float('inf')
best_config = None
sweep_results = []

USE_SILVER = False

# 4. Execute Hyperparameter Sweep
for run_id, config in enumerate(experiments, start=1):
    print(f"\n==================================================")
    print(f" SWEEP RUN {run_id}/{len(experiments)} | {config}")
    print(f"==================================================")

    set_seed(SEED)
    
    trainer = PropagandaTrainer(
        pipeline=pipeline,
        hidden_dim=config["hidden_dim"],
        dropout_p=config["dropout_p"],
        lr=config["lr"],
        use_silver=USE_SILVER    # vocab
    )

    save_filename = f"./param_sweep/sweep_gold_run_{run_id}.pt"

    epoch_tuples = trainer.run_training_loop(
        dataset_path='../data/propaganda_train.tsv',
        epochs=5,
        save_path=save_filename,
        save_best_only=True
    )

    # Capture overall top-performing configuration
    if trainer.best_val_loss < global_best_loss:
        global_best_loss = trainer.best_val_loss
        best_config = config
        print(f"🔥 NEW BEST MODEL FOUND! Dev Loss: {global_best_loss:.4f}")

    run_row = [
        run_id,
        config["hidden_dim"],
        config["lr"],
        config["dropout_p"],
        USE_SILVER,
        epoch_tuples  # Contains [(1, train_l, dev_l), (2, train_l, dev_l), ...]
    ]

    sweep_results.append(run_row)

print("\n" + "="*50)
print(f"SWEEP COMPLETE!")
print(f"Lowest Validation Loss: {global_best_loss:.4f}")
print(f"Optimal Hyperparameter Set: {best_config}")
print("="*50)

# ========================================================
# Save Sweep Results Directly to CSV File
# ========================================================
csv_filename = "./param_sweep/sweep_results_gold.csv"
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    # 1. Construct the header dynamically
    header = ["run_id", "hidden_dim", "lr", "dropout_p", "use_silver"]
    # Add columns for up to 5 epochs
    for i in range(1, 6):
        header.extend([f"train_loss_{i}", f"val_loss_{i}"])
    
    writer.writerow(header)
    
    # 2. Flatten the data for each run
    for row in sweep_results:
        run_id, hidden, lr, dropout, silver, history = row
        
        # Start the flat row with your metadata
        flat_row = [run_id, hidden, lr, dropout, silver]
        
        # Extract losses from each epoch tuple (epoch, train, val)
        for epoch_data in history:
            _, train_loss, val_loss = epoch_data
            flat_row.extend([train_loss, val_loss])
            
        writer.writerow(flat_row)

print(f"\nSweep complete, saved {len(sweep_results)} to '{csv_filename}'.")


Global seed set to 142
Vocabulary State Successfully Initialized.

 SWEEP RUN 1/12 | {'hidden_dim': 64, 'lr': 0.001, 'dropout_p': 0.3}
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.0650 | Avg Dev Loss: 2.0721
--> New best validation loss (2.0721) achieved! Model saved to ./param_sweep/sweep_gold_run_1.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.3971 | Avg Dev Loss: 1.9636
--> New best validation loss (1.9636) achieved! Model saved to ./param_sweep/sweep_gold_run_1.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.7827 | Avg Dev Loss: 2.1162
--> No improvement on validation loss. Skipping save.

--- Starting Epoch 4 ---
Epoch 4 Results | Avg Train Loss: 0.4696 | Avg Dev Loss: 2.4174
--> No improvement on validation loss. Skipping save.

--- Starting Epoch 5 ---
Epoch 5 Results | Avg Train Loss: 0.3612 | Avg Dev Loss: 2.5464
--> No improvement on validation loss. Skipping save.

🔥 NEW BEST MODEL FOUND! Dev Loss: 

In [117]:
HIDDEN_DIMS = 128
LR = 0.0001
DROPOUT = 0.5
EPOCHS = 3

##### BoW Baseline: Full Context, Gold Only

In [120]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = False

MODEL_SAVE = 'bow_full_gold'

bow_full_gold_pipeline = PropagandaFeaturePipeline()
bow_full_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_full_gold_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_full_gold_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_full_gold_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_full_gold_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_full_gold_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_full_gold_pipeline.ner_to_index)}")

bow_full_gold_eval_trainer = PropagandaTrainer(
    pipeline=bow_full_gold_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_full_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)

print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_full_gold_val_results = bow_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_full_gold_train_results = bow_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  3265
Dims of gold + silver sparse vec:  4002
Gold Only Singletons:              3038
Silver Enriched Singletons:        2301
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1974 | Avg Dev Loss: 2.0956
--> Model state updated at Epoch 1 and saved to ./final_models/bow_full_gold_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.7997 | Avg Dev Loss: 1.9858
--> Model state updated at Epoch 2 and saved to ./final_models/bow_full_gold_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.3216 | Avg Dev Loss: 1.9269
--> Model state updated at Epoch 3 and saved to ./final_models/bow_full_gold_weights.pt.

The results of the bow_full_gold model on the validation set:

 EVALUATION REPORT: ./final_models/bow_full_gold_weights.pt
 Accuracy:  0.3

##### BoW Baseline: Snippet, Gold Only

In [121]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False     # Snippet-only
USE_SILVER = False

MODEL_SAVE = 'bow_snippet_gold'

bow_snippet_gold_pipeline = PropagandaFeaturePipeline()

bow_snippet_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_snippet_gold_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_snippet_gold_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_snippet_gold_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_snippet_gold_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_snippet_gold_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_snippet_gold_pipeline.ner_to_index)}")

bow_snippet_gold_eval_trainer = PropagandaTrainer(
    pipeline=bow_snippet_gold_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_snippet_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)

print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_snippet_gold_val_results = bow_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_snippet_gold_train_results = bow_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  1483
Dims of gold + silver sparse vec:  2415
Gold Only Singletons:              2051
Silver Enriched Singletons:        1119
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1714 | Avg Dev Loss: 2.0250
--> Model state updated at Epoch 1 and saved to ./final_models/bow_snippet_gold_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.8766 | Avg Dev Loss: 1.9603
--> Model state updated at Epoch 2 and saved to ./final_models/bow_snippet_gold_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.6214 | Avg Dev Loss: 1.8933
--> Model state updated at Epoch 3 and saved to ./final_models/bow_snippet_gold_weights.pt.

The results of the bow_snippet_gold model on the validation set:

 EVALUATION REPORT: ./final_models/bow_snippet_gold_weights.pt


##### BoW Baseline: Full Context, Silver Enriched

In [122]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = True

MODEL_SAVE = 'bow_full_silver'

bow_full_silver_pipeline = PropagandaFeaturePipeline()

bow_full_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_full_silver_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_full_silver_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_full_silver_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_full_silver_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_full_silver_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_full_silver_pipeline.ner_to_index)}")

bow_full_silver_eval_trainer = PropagandaTrainer(
    pipeline=bow_full_silver_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_full_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)


print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_full_silver_val_results = bow_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_full_silver_train_results = bow_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  3265
Dims of gold + silver sparse vec:  4002
Gold Only Singletons:              3038
Silver Enriched Singletons:        2301
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1956 | Avg Dev Loss: 2.1225
--> Model state updated at Epoch 1 and saved to ./final_models/bow_full_silver_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.7399 | Avg Dev Loss: 2.0029
--> Model state updated at Epoch 2 and saved to ./final_models/bow_full_silver_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.1995 | Avg Dev Loss: 1.9584
--> Model state updated at Epoch 3 and saved to ./final_models/bow_full_silver_weights.pt.

The results of the bow_full_silver model on the validation set:

 EVALUATION REPORT: ./final_models/bow_full_silver_weights.pt
 Accu

##### BoW Baseline: Snippet, Silver Enriched

In [123]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False
USE_SILVER = True

MODEL_SAVE = 'bow_snippet_silver'

bow_snippet_silver_pipeline = PropagandaFeaturePipeline()

bow_snippet_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

print(f"Dims of baseline gold sparse vec:  {len(bow_snippet_silver_pipeline.word_to_index)}")
print(f"Dims of gold + silver sparse vec:  {len(bow_snippet_silver_pipeline.word_to_index_silver)}")
print(f"Gold Only Singletons:              {len(bow_snippet_silver_pipeline.hapax_words_list)}")
print(f"Silver Enriched Singletons:        {len(bow_snippet_silver_pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector:                {len(bow_snippet_silver_pipeline.pos_to_index)}")
print(f"Dimes of NER vector:               {len(bow_snippet_silver_pipeline.ner_to_index)}")

bow_snippet_silver_eval_trainer = PropagandaTrainer(
    pipeline=bow_snippet_silver_pipeline,
    hidden_dim=HIDDEN_DIMS,
    dropout_p=DROPOUT,
    lr = LR,
    weight_decay=0.05,
    use_silver=USE_SILVER
)

# training pass
set_seed(SEED)
bow_snippet_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False    # save final
)

print(f"The results of the {MODEL_SAVE} model on the validation set:")

bow_snippet_silver_val_results = bow_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'==='*10}")
print(f"")

print(f"The results of the {MODEL_SAVE} model on the training set:")

bow_snippet_silver_train_results = bow_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)

print(f"{'==='*10}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Dims of baseline gold sparse vec:  1483
Dims of gold + silver sparse vec:  2415
Gold Only Singletons:              2051
Silver Enriched Singletons:        1119
Dims of POS vector:                12
Dimes of NER vector:               10
Global seed set to 142
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 2.1746 | Avg Dev Loss: 2.0444
--> Model state updated at Epoch 1 and saved to ./final_models/bow_snippet_silver_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.7953 | Avg Dev Loss: 1.9664
--> Model state updated at Epoch 2 and saved to ./final_models/bow_snippet_silver_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 1.4224 | Avg Dev Loss: 1.9426
--> Model state updated at Epoch 3 and saved to ./final_models/bow_snippet_silver_weights.pt.

The results of the bow_snippet_silver model on the validation set:

 EVALUATION REPORT: ./final_models/bow_snippet_silver_w

#### Word2Vec Experiments

##### Hyperparameter Sweep: Word2Vec

In [169]:
import itertools
import csv

# 1. Pipeline Initialization & Vocab Construction
set_seed(SEED)

w2v_sweep_pipeline = PropagandaFeaturePipeline()
w2v_sweep_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=True
)

# 2. Search Grid Specifically Tailored for Dense Word2Vec Embeddings
param_grid = {
    'hidden_dim': [64, 128],
    'lr': [0.005, 0.001, 0.0005],
    'dropout_p': [0.3, 0.5]
}

# 3. Cartesian Product Combinations
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Track Metrics
global_best_loss = float('inf')
best_config = None
w2v_sweep_results = []

USE_SILVER = False
FEATURE_TYPE = "word2vec"
EPOCHS = 5

# 4. Execute Sweep
for run_id, config in enumerate(experiments, start=1):
    print(f"\n==================================================")
    print(f" WORD2VEC SWEEP RUN {run_id}/{len(experiments)} | {config}")
    print(f"==================================================")

    # Lock seed per run so every network configuration starts with identical weight initialization
    set_seed(SEED)
    
    trainer = PropagandaTrainer(
        pipeline=w2v_sweep_pipeline,
        hidden_dim=config["hidden_dim"],
        dropout_p=config["dropout_p"],
        lr=config["lr"],
        use_silver=USE_SILVER,
        feature_type=FEATURE_TYPE
    )

    save_filename = f"./param_sweep/sweep_w2v_gold_run_{run_id}.pt"

    epoch_tuples = trainer.run_training_loop(
        dataset_path='../data/propaganda_train.tsv',
        epochs=EPOCHS,
        save_path=save_filename,
        save_best_only=True
    )

    # Capture overall top-performing configuration
    if trainer.best_val_loss < global_best_loss:
        global_best_loss = trainer.best_val_loss
        best_config = config
        print(f"🔥 NEW BEST WORD2VEC MODEL FOUND! Dev Loss: {global_best_loss:.4f}")

    run_row = [
        run_id,
        config["hidden_dim"],
        config["lr"],
        config["dropout_p"],
        USE_SILVER,
        epoch_tuples  # [(epoch, train_loss, dev_loss), ...]
    ]

    w2v_sweep_results.append(run_row)

print("\n" + "="*50)
print(f"WORD2VEC SWEEP COMPLETE!")
print(f"Lowest Validation Loss: {global_best_loss:.4f}")
print(f"Optimal Hyperparameter Set: {best_config}")
print("="*50)

Global seed set to 142
Vocabulary State Successfully Initialized.

 WORD2VEC SWEEP RUN 1/12 | {'hidden_dim': 64, 'lr': 0.005, 'dropout_p': 0.3}
Global seed set to 142
Word2Vec model successfully linked from global scope.
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1256 | Avg Dev Loss: 2.1069
--> New best validation loss (2.1069) achieved! Model saved to ./param_sweep/sweep_w2v_gold_run_1.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 2.0179 | Avg Dev Loss: 2.0426
--> New best validation loss (2.0426) achieved! Model saved to ./param_sweep/sweep_w2v_gold_run_1.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.9436 | Avg Dev Loss: 2.0212
--> New best validation loss (2.0212) achieved! Model saved to ./param_sweep/sweep_w2v_gold_run_1.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.8882 |

In [174]:
W2W_HIDDEN_DIMS = 64
W2W_LR = 0.0005
W2W_DROPOUT = 0.5
W2W_EPOCHS = 5

##### Word2Vec: Full Context, Gold Only

In [175]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = False
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_full_gold'

# 1. Initialize Pipeline & Build Vocabularies
w2v_full_gold_pipeline = PropagandaFeaturePipeline()
w2v_full_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_full_gold_eval_trainer = PropagandaTrainer(
    pipeline=w2v_full_gold_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_full_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_full_gold_val_results = w2v_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_full_gold_train_results = w2v_full_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1292 | Avg Dev Loss: 2.0802
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_full_gold_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9767 | Avg Dev Loss: 2.0237
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_full_gold_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8775 | Avg Dev Loss: 1.9840
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_full_gold_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7903 | Avg Dev Loss: 1.9405
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_full_gold_weights.pt.

--- S

##### Word2Vec: Snippet, Gold Only

In [176]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False
USE_SILVER = False
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_snippet_gold'

# 1. Initialize Pipeline & Build Vocabularies
w2v_snippet_gold_pipeline = PropagandaFeaturePipeline()
w2v_snippet_gold_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_snippet_gold_eval_trainer = PropagandaTrainer(
    pipeline=w2v_snippet_gold_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_snippet_gold_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_snippet_gold_val_results = w2v_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_snippet_gold_train_results = w2v_snippet_gold_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1304 | Avg Dev Loss: 2.0899
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_snippet_gold_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9802 | Avg Dev Loss: 2.0189
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_snippet_gold_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8753 | Avg Dev Loss: 1.9670
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_snippet_gold_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7833 | Avg Dev Loss: 1.9213
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_snippet_gold_weight

##### Word2Vec: Full Context, Silver Enriched

In [177]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = True
USE_SILVER = True
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_full_silver'

# 1. Initialize Pipeline & Build Vocabularies
w2v_full_silver_pipeline = PropagandaFeaturePipeline()
w2v_full_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_full_silver_eval_trainer = PropagandaTrainer(
    pipeline=w2v_full_silver_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_full_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_full_silver_val_results = w2v_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_full_silver_train_results = w2v_full_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1302 | Avg Dev Loss: 2.0805
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_full_silver_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9815 | Avg Dev Loss: 2.0229
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_full_silver_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8802 | Avg Dev Loss: 1.9751
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_full_silver_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7937 | Avg Dev Loss: 1.9083
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_full_silver_weights.pt

##### Word2Vec: Snippet, Silver Enriched

In [178]:
import os
model_dir = './final_models'
os.makedirs(model_dir, exist_ok=True)

set_seed(SEED)

FULL_CONTEXT = False
USE_SILVER = True
FEATURE_TYPE = "word2vec"
MODEL_SAVE = 'w2v_snippet_silver'

# 1. Initialize Pipeline & Build Vocabularies
w2v_snippet_silver_pipeline = PropagandaFeaturePipeline()
w2v_snippet_silver_pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv',
    full_context=FULL_CONTEXT
)

# 2. Instantiate Trainer with Word2Vec feature type
w2v_snippet_silver_eval_trainer = PropagandaTrainer(
    pipeline=w2v_snippet_silver_pipeline,
    hidden_dim=W2W_HIDDEN_DIMS,
    dropout_p=W2W_DROPOUT,
    lr=W2W_LR,
    weight_decay=0.05,
    use_silver=USE_SILVER,
    feature_type=FEATURE_TYPE
)

# 3. Training Pass
set_seed(SEED)
w2v_snippet_silver_eval_trainer.run_training_loop(
    dataset_path='../data/propaganda_train.tsv',
    epochs=W2W_EPOCHS,
    save_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    save_best_only=False
)

# 4. Evaluation
print(f"The results of the {MODEL_SAVE} model on the validation set:")
w2v_snippet_silver_val_results = w2v_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_val.tsv'
)

print(f"{'===='+'===='+'=='}\n")
print(f"The results of the {MODEL_SAVE} model on the training set:")
w2v_snippet_silver_train_results = w2v_snippet_silver_eval_trainer.evaluate(
    weights_path=f'./final_models/{MODEL_SAVE}_weights.pt',
    dataset_path='../data/propaganda_train.tsv'
)
print(f"{'===='+'===='+'=='}")

Global seed set to 142
Vocabulary State Successfully Initialized.
Word2Vec model successfully linked from global scope.
Global seed set to 142
--- Starting Epoch 1 ---
empty w2v vector
empty w2v vector
Epoch 1 Results | Avg Train Loss: 2.1255 | Avg Dev Loss: 2.0810
--> Model state updated at Epoch 1 and saved to ./final_models/w2v_snippet_silver_weights.pt.

--- Starting Epoch 2 ---
empty w2v vector
empty w2v vector
Epoch 2 Results | Avg Train Loss: 1.9715 | Avg Dev Loss: 1.9991
--> Model state updated at Epoch 2 and saved to ./final_models/w2v_snippet_silver_weights.pt.

--- Starting Epoch 3 ---
empty w2v vector
empty w2v vector
Epoch 3 Results | Avg Train Loss: 1.8633 | Avg Dev Loss: 1.9484
--> Model state updated at Epoch 3 and saved to ./final_models/w2v_snippet_silver_weights.pt.

--- Starting Epoch 4 ---
empty w2v vector
empty w2v vector
Epoch 4 Results | Avg Train Loss: 1.7724 | Avg Dev Loss: 1.8982
--> Model state updated at Epoch 4 and saved to ./final_models/w2v_snippet_silve

## Task 2

#### Installations

In [1]:
# ==========================================
# CELL 1: Installations
# ==========================================
%pip install -q transformers sentencepiece
%pip install -q pytorch-crf
%pip install -q scikit-learn
%pip install -q tiktoken
%pip install protobuf sentencepiece


[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports & Global Configuration

In [231]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import re
import csv
import torch
import random
import string
import numpy as np
import torch.nn as nn
from transformers import AutoTokenizer, DebertaV2Model
from torchcrf import CRF
from sklearn.metrics import precision_recall_fscore_support, classification_report

# Define global device for M1 GPU acceleration ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed: int = 142):
    """Seed Logic for Reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

SEED = 142
set_seed(SEED)

# Task 2 Label Schemas
TECHNIQUES = [
    'flag_waving', 'appeal_to_fear_prejudice', 'causal_oversimplification', 
    'doubt', 'loaded_language', 'name_calling,labeling', 
    'repetition', 'exaggeration,minimisation'
]

# Variation 1 Tagset (3-Class Boundary Detection)
BIO_3_CLASS = ['O', 'B-Propaganda', 'I-Propaganda']
V1_TAG_TO_IDX = {tag: i for i, tag in enumerate(BIO_3_CLASS)}
V1_IDX_TO_TAG = {i: tag for tag, i in V1_TAG_TO_IDX.items()}

# Variation 2 Tagset (17-Class Joint Detection)
BIO_17_CLASS = ['O'] + [f"B-{t}" for t in TECHNIQUES] + [f"I-{t}" for t in TECHNIQUES]
V2_TAG_TO_IDX = {tag: i for i, tag in enumerate(BIO_17_CLASS)}
V2_IDX_TO_TAG = {i: tag for tag, i in V2_TAG_TO_IDX.items()}

# Classifier Head Tagset
TECH_TO_IDX = {tech: i for i, tech in enumerate(TECHNIQUES)}
IDX_TO_TECH = {i: tech for tech, i in TECH_TO_IDX.items()}

Using device: mps
Global seed set to 142


#### Data Processing Class

In [243]:
class Task2DataPipeline:
    """
    Processes raw TSV rows into DeBERTa subword encodings and aligned BIO tags.
    Handles <BOS> and <EOS> markers present across all rows (including not_propaganda).
    """
    def __init__(self, model_checkpoint="microsoft/deberta-v3-xsmall"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        
    def universal_cleaning_t2(self, raw_text: str) -> str:
        """Cleans text while preserving case and punctuation for Transformer context."""
        text = raw_text.strip()
        text = text.replace("\\'", "'").replace('\\"', '"') # Escape characters
        text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
        text = re.sub(r'[\\/\[\]*|@$#+=]', ' ', text) # Strip digital artifacts
        return " ".join(text.split())

    def parse_span_and_clean(self, raw_text: str, label: str):
        """
        Extracts exact character offsets of the snippet bounded by <BOS>/<EOS>.
        Every row in the dataset contains <BOS>/<EOS> tags.
        is_active_propaganda is True ONLY if the row's label is NOT 'not_propaganda'.
        """
        text = self.universal_cleaning_t2(raw_text)
        
        bos_idx = text.find("<BOS>")
        eos_idx = text.find("<EOS>")
        
        # Where tag boundaries are found, extract the exact character indices
        if bos_idx != -1 and eos_idx != -1:
            pre_bos = text[:bos_idx]
            inside = text[bos_idx+5:eos_idx]    # +5 to skip over '<BOS>' string length
            post_eos = text[eos_idx+5:]         # +5 to skip over '<EOS>' string length
            
            clean_text = pre_bos + inside + post_eos    # no tags
            char_start = len(pre_bos)                   # character index where snippet starts
            char_end = len(pre_bos) + len(inside)       # character index where snippet end
            
            is_active_propaganda = 1.0 if label != 'not_propaganda' else 0.0
        else:
            # Fallback if input data is incorrecct
            print("ERROR: Instance has no tags")
            clean_text = text.replace("<BOS>", "").replace("<EOS>", "")
            char_start, char_end = -1, -1
            is_active_propaganda = 0.0
            
        return clean_text, char_start, char_end, is_active_propaganda

    def align_bio_tags(self, clean_text: str, char_start: int, char_end: int, label: str, mode: str):
        """
        Tokenizes clean_text and aligns subword offset mappings to character spans.
        self.tokenizer = AutoTokenizer
        If label == 'not_propaganda', all tokens are assigned the 'O' tag (ID 0).
        """

        # SentencePeice DeBERTA native subword tokenizer
        encoding = self.tokenizer(
            clean_text, 
            return_offsets_mapping=True,    # char location of tokenz
            truncation=True, 
            max_length=512,
            return_tensors="pt"
        )
        
        # Obtain the token to character indices mappings: [(start,end)]
        offsets = encoding['offset_mapping'][0].tolist()
        
        tag_ids = []
        is_in_span = False
        
        for idx, (o_start, o_end) in enumerate(offsets):
            if o_start == o_end: # zero-length tokens or special boundary: [CLS], [SEP], Padding
                tag_ids.append(0)
                continue
                
            # Assign B-/I- tags ONLY if label is an active propaganda technique
            if (label != 'not_propaganda' and 
                char_start != -1 and    # real token only
                char_start <= o_start   # start of token is within span start
                and o_end <= char_end   # end of token is within span end
                ):

                if not is_in_span:  # First instance of propaganda
                    tag = "B-Propaganda" if mode == "var1" else f"B-{label}"
                    is_in_span = True   # Update tracker
                else:
                    tag = "I-Propaganda" if mode == "var1" else f"I-{label}"
                    is_in_span = True
                
                tag_map = V1_TAG_TO_IDX if mode == "var1" else V2_TAG_TO_IDX
                tag_ids.append(tag_map.get(tag, 0))     # convert tags to numerical ID

                # this tagging process will contine appending I- until `o_end <= char_end` is breached

            else:
                tag_ids.append(0) # 'O' tag for non-propaganda text or outside span
                is_in_span = False

        input_ids = encoding['input_ids']                           # map to SP vocabulary
        attention_mask = encoding['attention_mask']                 # 1's for words, 0 for specials [SEP, CLS, Padding]
        tags_tensor = torch.tensor([tag_ids], dtype=torch.long)     # convert to pytorch format
        
        # Ensure input_ids, attention_mask, and tag_ids match in sequence length
        assert input_ids.shape[1] == attention_mask.shape[1] == tags_tensor.shape[1], (
            f"Dimension mismatch! input_ids: {input_ids.shape[1]}, "
            f"attention_mask: {attention_mask.shape[1]}, tags: {tags_tensor.shape[1]}"
        )
                
        return input_ids, attention_mask, tags_tensor

#### Evaluation Logic Class

In [260]:
# ==========================================
# CELL 4: Cascading Window Qualification Router (Evaluation)
# ==========================================

class Task2Evaluator:
    
    @staticmethod
    def get_tolerance(span_length: int) -> int:
        """helper function for span evaluation router with slightly increased tolerance"""
        if span_length <= 5: return 1
        elif span_length <= 10: return 2
        elif span_length <= 15: return 3
        elif span_length <= 50: return 3 + ((span_length - 11) // 5)
        else: return 12

    @staticmethod
    def evaluate_predictions(gold_data: list, pred_data: list):
        """
        input gold_data/pred_data format: [{"span": (start_idx, end_idx), "technique": "doubt"}, ...]
        
        Returns:
            - Tuple: (metrics_dict, classification_report_str)
        """
        y_true = []
        y_pred = []
        error_logs = []  # verbose capture of errors
        
        for gold, pred in zip(gold_data, pred_data):
            
            # ==========================================
            # --- Condition 1: True Negative (TN)    ---
            # ---  * Gold instance is not_propaganda ---
            # ---  * Model did not predict a span    ---
            # ==========================================
            if gold["technique"] == "not_propaganda" and pred["span"] == (-1, -1):
                continue  # True Negative, ignored in Macro-F1
            
            # ==========================================
            # --- Condition 2: False Positive (FP)   ---
            # ---  * Gold instance is not_propaganda ---
            # ---  * Model predicted a span    ---
            # ==========================================
            if gold["technique"] == "not_propaganda" and pred["span"] != (-1, -1):
                y_true.append("not_propaganda")
                y_pred.append(pred["technique"])
                error_logs.append({"error": "Hallucinated Span", "pred": pred})
                continue
            
            # ==========================================
            # --- Condition 3: False Negative (FN)   ---
            # ---  * Gold instance is propaganda     ---
            # ---  * Model did not predict a span    ---
            # ==========================================
            if gold["technique"] != "not_propaganda" and pred["span"] == (-1, -1):
                y_true.append(gold["technique"])
                y_pred.append("not_propaganda")
                error_logs.append({"error": "Missed Span", "gold": gold})
                continue

            # =============================================
            # --- Condition 4: Cascading Window Router  ---
            # --- Potential for True Positive (TP)      ---
            # ---  * Gold instance is propaganda        ---
            # ---  * model predicted an active span     ---
            # =============================================
            g_start, g_end = gold["span"]
            p_start, p_end = pred["span"]
            gold_len = g_end - g_start + 1      # +1 for inclusive bounds
            delta = Task2Evaluator.get_tolerance(gold_len)
            
            # =================================================
            # --- Boundary Qualified                        ---
            # --- True Positive (TP) or False Positive (FP) ---
            # =================================================
            if abs(p_start - g_start) <= delta and abs(p_end - g_end) <= delta:
                y_true.append(gold["technique"])    # Capacity for True Positive (TP)
                y_pred.append(pred["technique"])    # If the prediction is wrong then it appends a False Positive (FP)
                if gold["technique"] != pred["technique"]:  
                    # Capacity for FP/FN despite qualifying: Span correct but wrong technique predicted
                    error_logs.append({"error": "Technique Misclassification", "gold": gold, "pred": pred})

            # ===================================================
            # --- Boundary Disqualified (Double Penalty)      ---
            # --- False Positive (FP) and False Negative (FN) ---
            # ===================================================
            else:
                left_failed = abs(p_start - g_start) > delta
                right_failed = abs(p_end - g_end) > delta
                
                # Logging Subtype Failure: Fail Side
                if left_failed and right_failed:
                    failure_subtype = "Boundary Failure: Both Left and Right"
                elif left_failed:
                    failure_subtype = "Boundary Failure: Left Only (Start Offset)"
                else:
                    failure_subtype = "Boundary Failure: Right Only (End Offset)"

                # ===================================
                # --- Double Counting the Failure --- 
                # ===================================
                y_true.append(gold["technique"])
                y_pred.append("not_propaganda")  # FN for target
                # A failing span is the equivalent to a not_propaganda prediction 
                # As it couldnt find the **correct** span
                # But there was a target which needs which needs a False Negative mark
                # This impacts Recall, as a target was missed.
                
                y_true.append("not_propaganda")
                y_pred.append(pred["technique"])  # FP for prediction
                # A span was predicted at an invalid location
                # An in invalid location must have a label of not_propaganda
                # The model the assigned this invalid location a technique
                # This impacts Precision, as a target was hallucinated
                
                error_logs.append({
                    "error": "Boundary Localization Failure", 
                    "subtype": failure_subtype,
                    "gold": gold, 
                    "pred": pred,
                    "delta_allowed": delta,
                    "left_diff": abs(p_start - g_start),
                    "right_diff": abs(p_end - g_end)
                })

        # TODO: `print_diagnostic_breakdown`

        # Terminal aggregate metrics (Macro-F1 ignoring 'not_propaganda')
        valid_labels = TECHNIQUES
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=valid_labels, average='macro', zero_division=0
        )
        
        # Per-class metrics via scikit-learn
        class_report = classification_report(
            y_true, y_pred, labels=TECHNIQUES, zero_division=0
        )
        
        metrics_dict = {
            "Macro-F1": f1, 
            "Precision": precision, 
            "Recall": recall, 
            "Error_Logs": error_logs,
            "y_true": y_true,
            "y_pred": y_pred
        }

        return metrics_dict, class_report

#### Architecture Class

In [245]:
class DebertaCRFTagger(nn.Module):
    """
    - A torch.nn.Module to define the structural blueprint and math operations
    - Acccess data dynamically when used in training or evaluaiton loops
    - Forward pass accepts input_ids, attention_mask, and optionally tags
    - Tagger class itself only takes num_tags as an arg to route to Variation 1 or 2

    - [DeBERTa Encoder]  -->  [Linear Projection Layer]  -->  [Linear-Chain CRF]
    - (DebertaV2Model)   --> (self.hidden2tag)           --> (self.crf)
    
    - Tagger converts raw text subwords into rich, context-aware vector representations (hidden states)
    - Dense vector of size: 384 (for microsoft/deberta-v3-xsmall)
    - DeBERTa is pre-trained but is fine-tuned via backpropagation as the weights are not frozen. 

    - Linear Projection Layer takes 384 dims -> num_tags emissions via self.hidden2tag
    - Converts continuous semantic vectors into raw category emission scores for each tag at each token position.
    - This is trained via backpropagation

    - Passes emission scores into CRF layer for sequence decoding/loss
    - CRF holds a trainable transition matrix ($\text{num\_tags} \times \text{num\_tags}$) to learn sequence rules across tags.
    - Enforces sequence-level dependencies and transition rules.
    - This is trained via backpropagation to learn which tag transitions are likely and which are impossible
    
    """
    def __init__(self, mode="var2", model_checkpoint="microsoft/deberta-v3-xsmall"):
        super().__init__()
        self.mode = mode

        # Init Base Model
        self.deberta = DebertaV2Model.from_pretrained(model_checkpoint).float() # contextualised vectors
        hidden_size = self.deberta.config.hidden_size # 384
        print("DeBERTa Hidden Size:", hidden_size)

        # ============================================================
        # Task 2 Variation Router                                  ===
        #   * Var 1: Seperate Heads for Classification & Boundary  ===
        #   * Var 2: Integrated Head for Classification & Boundary ===
        # ============================================================
        if mode == "var1":
            # Head 1: 3-Class Boundary Detection [O, B-Prop, I-Prop]
            self.boundary_head = nn.Linear(hidden_size, 3)
        
            # Head 2: 8-Class Technique MLP Classification
            self.technique_head = nn.Sequential(
                nn.Linear(hidden_size, 64),
                nn.ReLU(),
                nn.LayerNorm(64),
                nn.Dropout(0.3),
                nn.Linear(64, 8)
            )
        else:
            # Single 17-Class Joint Linear Mapping
            self.hidden2tag = nn.Linear(hidden_size, 17)
        
        # CRF Layer
        self.crf = CRF(17, batch_first=True) # Both vars converge to 17 tagset for CRF

        # CRF Overrides/Constraints
        #   - Rule 1: Always O to B
        #   - Rule 2: Never switch techniques
        self.apply_bio_constraints()


    def apply_bio_constraints(self):
        """Enforces 17-class hard BIO transition constraints (-10000.0)."""
        idx_to_tag = V2_IDX_TO_TAG
        tag_to_idx = {tag: idx for idx, tag in idx_to_tag.items()}
        
        # Access the raw CRF transition parameter matrix
        # self.crf.transitions[i, j] represents score of moving FROM tag i TO tag j
        with torch.no_grad():
            for from_tag, i in tag_to_idx.items():
                for to_tag, j in tag_to_idx.items():
                    # Rule 1: FORBID O -> I-tag (Must enter a span through a B- tag)
                    if from_tag == "O" and to_tag.startswith("I-"):
                        self.crf.transitions[i, j] = -10000.0
                    # RULE 2: FORBID mid-span technique switching!
                    # I-tech_A or B-tech_A can ONLY transition to I-tech_A (same technique)
                    if (from_tag.startswith("B-") or from_tag.startswith("I-")) and to_tag.startswith("I-"):
                        if from_tag.split("-")[1] != to_tag.split("-")[1]:
                            self.crf.transitions[i, j] = -10000.0

    def _compile_17_tag_logits(self, boundary_logits, tech_logits):
        """Broadcasts 3-class boundary scores and 8-class technique scores into 17-class BIO logits."""
        o_logit = boundary_logits[:, :, 0:1]       # Index 0: O
        b_prop_logit = boundary_logits[:, :, 1:2]  # Index 1: B-Propaganda
        i_prop_logit = boundary_logits[:, :, 2:3]  # Index 2: I-Propaganda
        
        b_tech_logits = b_prop_logit + tech_logits # Broadcast addition -> 8 B-Technique logits
        i_tech_logits = i_prop_logit + tech_logits # Broadcast addition -> 8 I-Technique logits
        
        return torch.cat([o_logit, b_tech_logits, i_tech_logits], dim=-1)

    def forward(self, input_ids, attention_mask, tags=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state     # 384 dims for each token: (batch_size, sequence_length, hidden_size)
        
        # --- LOGIT COMPILATION ---
        # projects each 384-dimensional token vector down to num_tags scores
        if self.mode == "var1":
            boundary_logits = self.boundary_head(sequence_output)
            tech_logits = self.technique_head(sequence_output)
            emissions = self._compile_17_tag_logits(boundary_logits, tech_logits)
        else:
            emissions = self.hidden2tag(sequence_output)

        # --- UNIFIED CRF PASS ---
        if tags is not None:
            # Training: 
            # uses the provided 'tags' to compute the Negative Log-Likelihood Loss
            return -self.crf(emissions, tags, mask=attention_mask.byte(), reduction='mean')
        else:
            # Inference: 
            # Viterbi decoding to predict tags, best sequence per input sentence
            return self.crf.decode(emissions, mask=attention_mask.byte())


class SpanClassifierHead(nn.Module):
    """Contains the classifer head required for Variation 1"""
    def __init__(self, input_dim=384, hidden_dim=64, num_classes=8): # 384 for deberta-v3-xsmall
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, span_embedding):
        return self.mlp(span_embedding)

<>:2: SyntaxWarning: invalid escape sequence '\_'
<>:2: SyntaxWarning: invalid escape sequence '\_'
/var/folders/q9/kmgt4jcs40qbkm14m7p96ld40000gn/T/ipykernel_21374/1871888700.py:2: SyntaxWarning: invalid escape sequence '\_'
  """


#### Execution Class (Training, Eval, Viterbi)

In [257]:
# ==========================================
# CELL 6: Execution Engine (Updated calls)
# ==========================================

import itertools
from sklearn.metrics import classification_report

class Task2Executor:
    """This is the class that orchestrates the models pipelines:
        - `Task2DataPipeline` prepares raw text
        - `DebertaCRFTagger` defines the architecture
        - `Task2Executor` utilises these components
    """
    def __init__(self, pipeline: Task2DataPipeline):
        self.pipeline = pipeline
        

    def extract_viterbi_span(self, viterbi_path, mode="var2"):
        """
        Converts raw CRF tag integer sequences into token span boundaries

        - Takes the raw sequence of predicted integer tag IDs (viterbi_path) returned by the CRF decoder
        - Converts tag IDs to BIO string names using V1_IDX_TO_TAG (3 tags) or V2_IDX_TO_TAG (17 tags)
        - Extracts the first continuous non-O span and its start token index, end token index, and technique label
        Outputs are evaluated against gold-standard annotations
        """
        start_idx, end_idx, technique = -1, -1, "not_propaganda"    # default, overwrite if propaganda
        
        idx_to_tag = V2_IDX_TO_TAG # Both vars converge to this tagset
        
        for i, tag_id in enumerate(viterbi_path):
            tag = idx_to_tag[tag_id]
            if tag.startswith("B-"):
                start_idx = i
                end_idx = i
                technique = tag.split("-")[1]
            elif tag.startswith("I-") and start_idx != -1:
                end_idx = i
                
        # Ensure flat integers are always returned
        if start_idx == -1:
            return -1, -1, "not_propaganda"
            
        return start_idx, end_idx, technique
    

    def train_deberta_tagger(
        self, 
        train_path, 
        mode="var2", 
        epochs=5,
        batch_size=16, 
        backbone_lr=1e-5,
        heads_lr=5e-4 
    ):
        print(f"\n--- Training DeBERTa-CRF ({mode.upper()}) [Batch Size: {batch_size}] ---")
        model = DebertaCRFTagger(mode=mode).to(device)

        # --- DYNAMIC OPTIMIZER ROUTING ---
        if mode == "var1":
            head_params = list(model.boundary_head.parameters()) + \
                          list(model.technique_head.parameters()) + \
                          list(model.crf.parameters())
        else:
            head_params = list(model.hidden2tag.parameters()) + \
                          list(model.crf.parameters())

        # Use the passed parameters here:
        optimizer = torch.optim.AdamW([
            {'params': model.deberta.parameters(), 'lr': backbone_lr},
            {'params': head_params, 'lr': heads_lr}
        ])
                
        accum_steps = batch_size
        model.train()
        
        for epoch in range(epochs):
            total_loss = 0.0
            with open(train_path, mode='r', encoding='utf-8') as file:
                reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
                optimizer.zero_grad()
                
                for i, row in enumerate(reader, start=1):

                    # Feature Extraction
                    label = row['label']
                    clean_text, c_start, c_end, _ = self.pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                    
                    # Always generate 17-class target IDs for both Var1 & Var2
                    input_ids, att_mask, tags = self.pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode="var2")

                    # MAC GPU
                    input_ids = input_ids.to(device)
                    att_mask = att_mask.to(device)
                    tags = tags.to(device)
                    
                    # Backprop
                    loss = model(input_ids, att_mask, tags)
                    scaled_loss = loss / accum_steps
                    scaled_loss.backward()
                    total_loss += loss.item()

                    # Batching
                    if i % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                        optimizer.zero_grad()
                    
                    if i % 100 == 0:
                        print(f"  Processed {i} instances...")
                
                # Residual Gradients outside of final modulo
                if i % accum_steps != 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()
                        
            print(f"Epoch {epoch+1} | Total CRF NLL Loss: {total_loss:.4f}")
            
        return model
        

    def evaluate_tagger(self, model, test_path, mode="var2"):
        """
        Runs inference over evaluation data and builds evaluation dictionaries

        - Executes model inference and formats predictions for metric calculation
        - Processes raw text: parse_span_and_clean and align_bio_tags
        - Using extract_viterbi_span on the gold data to obtain gold label spans
        - Calls model(input_ids, att_mask) without tags triggering CRF's Viterbi decoder
        - Returns the optimal predicted tag ID sequence (viterbi_path)
        - extract_viterbi_span() again to get predicted spans
        - Formats predictions into gold_data and pred_data lists of dicts
        - delegates evaluation to Task2Evaluator.evaluate_predictions 

        """
        model.eval()
        gold_data, pred_data = [], []
        
        with open(test_path, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in reader:

                # Gold Data: extract label, clean text, tokenize & BIO tag, capture span indices. 
                label = row['label']
                clean_text, c_start, c_end, _ = self.pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                input_ids, att_mask, tags = self.pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode=mode)
                g_start, g_end, _ = self.extract_viterbi_span(tags[0].tolist(), mode=mode)

                # Move DeBERTa Inputs to M1 GPU
                input_ids = input_ids.to(device)
                att_mask = att_mask.to(device)

                # Prediction Data: Viterbi Path 
                with torch.no_grad():
                    # trained model forward pass, extract most likely BIO sequence (IDs)
                    viterbi_path = model(input_ids, att_mask)[0]
                
                p_start, p_end, p_tech = self.extract_viterbi_span(viterbi_path, mode=mode)
                
                # Collect data
                gold_data.append({"span": (g_start, g_end), "technique": label})
                pred_data.append({"span": (p_start, p_end), "technique": p_tech})
    
                
        eval_results, class_report = Task2Evaluator.evaluate_predictions(gold_data, pred_data)
        return eval_results, class_report


#### Baseline Class

In [248]:
class Task2RandomBaseline:
    """
    Independent stochastic baseline for Task 2.
    Generates non-linguistic uniform random span boundaries and technique predictions
    """
    def __init__(self, pipeline: Task2DataPipeline, executor: Task2Executor):
        self.pipeline = pipeline
        self.executor = executor

    def _stochastic_generation(self, prop_threshold: float, num_tokens: int):
        """Generates random span boundaries and technique predictions."""
        if random.random() < prop_threshold:    
            # Snippet Boundary Random Guess
            if num_tokens > 3:
                p_start = random.randint(1, num_tokens - 2)         # -2 to avoid end SEP token
                p_end = random.randint(p_start, num_tokens - 2)     # limited to first pred onwards
            else:
                p_start, p_end = 1, 1   # fixed as middle guess for really short seqs
                
            # Technique Random Guess
            p_tech = random.choice(TECHNIQUES)  # tech is condictional random guess of 8 techs
        else:
            # Predict neutral sentence
            p_start, p_end = -1, -1
            p_tech = "not_propaganda"

        return p_start, p_end, p_tech

    def evaluate(self, test_path: str, dataset_name: str = "VAL", prop_threshold: float = 0.5):
        """ 
        Evaluator for the random baseline. 
        Largely mimics evaluate_tagger() but without the modelling complexity
        """
        print(f"\n--- Running Stochastic Random-Guessing Baseline ({dataset_name}) ---")
        gold_data, pred_data = [], []
        with open(test_path, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in reader:
                
                # Gold Data: extract label, clean text, tokenize & BIO tag, capture span indices. 
                label = row['label']
                clean_text, c_start, c_end, _ = self.pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                input_ids, att_mask, tags = self.pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode="var1")
                g_start, g_end, _ = self.executor.extract_viterbi_span(tags[0].tolist(), mode="var1")

                # Prediction Data: token length param, randomly generate spans and predictions
                actual_len = int(att_mask.sum().item())
                p_start, p_end, p_tech = self._stochastic_generation(prop_threshold, actual_len)

                # Collect data
                gold_data.append({"span": (g_start, g_end), "technique": label})
                pred_data.append({"span": (p_start, p_end), "technique": p_tech})
        
        # Evaluate results + class breakdown
        eval_results, class_report = Task2Evaluator.evaluate_predictions(gold_data, pred_data)
        
        return eval_results, class_report, gold_data, pred_data

In [249]:
def _propaganda_ratio(TRAIN_PATH):
    """A function to work out the training not_propaganda split for the random guesser"""
    active_count, total_count = 0, 0
    with open(TRAIN_PATH, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
        for row in reader:
            total_count += 1
            if row['label'] != 'not_propaganda':
                active_count += 1

    empirical_ratio = active_count / max(1, total_count)
    return empirical_ratio

#### Baseline Results

In [206]:
# ==========================================
# CELL 7: Task 2 Baseline Execution
# ==========================================

# 1. Lock seed for exact reproducibility
set_seed(SEED)

TRAIN_PATH = '../data/propaganda_train.tsv'
VAL_PATH = '../data/propaganda_val.tsv'

t2_pipeline = Task2DataPipeline()
executor = Task2Executor(t2_pipeline)
random_baseline = Task2RandomBaseline(t2_pipeline, executor)

# 4. Compute empirical training prior (proportion of active propaganda instances)
empirical_ratio = _propaganda_ratio(TRAIN_PATH)

# 5. Execute stochastic baseline evaluation on Validation/Test split
print("=" * 60)
print("     STARTING TASK 2 BASELINE EVALUATION")
print("=" * 60)

metrics, class_breakdown, baseline_gold_data, baseline_pred_data = random_baseline.evaluate(
    VAL_PATH, 
    dataset_name="TEST/VAL SET", 
    prop_threshold=empirical_ratio
)

print(f"\n==========================================")
print(f" BASELINE EVALUATION REPORT: {TRAIN_PATH}")
print(f"==========================================")
print(f" Macro-F1 Score:  {metrics['Macro-F1']:.4f}")
print(f" Macro Precision: {metrics['Precision']:.4f}")
print(f" Macro Recall:    {metrics['Recall']:.4f}")
print(f"------------------------------------------")
print(class_breakdown)
print(f"==========================================\n")


Global seed set to 142
     STARTING TASK 2 BASELINE EVALUATION

--- Running Stochastic Random-Guessing Baseline (TEST/VAL SET) ---

 BASELINE EVALUATION REPORT: ../data/propaganda_train.tsv
 Macro-F1 Score:  0.0000
 Macro Precision: 0.0000
 Macro Recall:    0.0000
------------------------------------------
                           precision    recall  f1-score   support

              flag_waving       0.00      0.00      0.00      45.0
 appeal_to_fear_prejudice       0.00      0.00      0.00      43.0
causal_oversimplification       0.00      0.00      0.00      35.0
                    doubt       0.00      0.00      0.00      43.0
          loaded_language       0.00      0.00      0.00      39.0
    name_calling,labeling       0.00      0.00      0.00      34.0
               repetition       0.00      0.00      0.00      40.0
exaggeration,minimisation       0.00      0.00      0.00      30.0

                micro avg       0.00      0.00      0.00     309.0
                mac

#### Baseline Analysis

In [212]:
# ==========================================
# Task 2 Baseline Analytical Diagnostics
# ==========================================

def analyze_random_baseline(gold_data, pred_data):
    total_samples = len(gold_data)
    
    # 1. Routing Breakdown
    routed_not_prop = [p for p in pred_data if p["span"] == (-1, -1)]
    routed_span = [p for p in pred_data if p["span"] != (-1, -1)]
    
    pct_not_prop = (len(routed_not_prop) / total_samples) * 100
    pct_span = (len(routed_span) / total_samples) * 100
    
    # 2. Composition of the `not_propaganda` routed subsample
    # Match indices back to gold labels
    not_prop_indices = [i for i, p in enumerate(pred_data) if p["span"] == (-1, -1)]
    actual_gold_not_prop = sum(1 for i in not_prop_indices if gold_data[i]["technique"] == "not_propaganda")
    actual_gold_is_prop = len(not_prop_indices) - actual_gold_not_prop
    
    pct_actually_not_prop = (actual_gold_not_prop / max(1, len(not_prop_indices))) * 100
    pct_actually_prop_missed = (actual_gold_is_prop / max(1, len(not_prop_indices))) * 100
    
    # 3. Composition of the Span Guess routed subsample
    span_indices = [i for i, p in enumerate(pred_data) if p["span"] != (-1, -1)]
    span_gold_prop = sum(1 for i in span_indices if gold_data[i]["technique"] != "not_propaganda")
    span_gold_not_prop = len(span_indices) - span_gold_prop
    
    pct_span_has_gold_target = (span_gold_prop / max(1, len(span_indices))) * 100
    
    # 4. Successful Boundary Passes (True Positives on Span)
    successful_spans = 0
    for i in span_indices:
        g = gold_data[i]
        p = pred_data[i]
        if g["technique"] != "not_propaganda":
            g_start, g_end = g["span"]
            p_start, p_end = p["span"]
            delta = Task2Evaluator.get_tolerance(g_end - g_start + 1)
            if abs(p_start - g_start) <= delta and abs(p_end - g_end) <= delta:
                successful_spans += 1
                # print(gold_data[i], pred_data[i])
                
    print("=" * 60)
    print("      TASK 2 RANDOM BASELINE DIAGNOSTIC ANALYSIS")
    print("=" * 60)
    print(f"Total Evaluation Instances : {total_samples}")
    print(f"------------------------------------------------------------")
    print(f"1. Routing Distribution:")
    print(f"   - Routed to 'not_propaganda' (-1, -1) : {len(routed_not_prop)} ({pct_not_prop:.2f}%)")
    print(f"   - Routed to Active Span Guess        : {len(routed_span)} ({pct_span:.2f}%)")
    print(f"------------------------------------------------------------")
    print(f"2. Subsample Composition ('not_propaganda' route):")
    print(f"   - Of those guessed as background, actually gold background : {pct_actually_not_prop:.2f}%")
    print(f"   - Of those guessed as background, actually missed targets   : {pct_actually_prop_missed:.2f}%")
    print(f"------------------------------------------------------------")
    print(f"3. Subsample Composition (Active Span route):")
    print(f"   - Of those guessed as spans, held a valid propaganda target : {pct_span_has_gold_target:.2f}%")
    print(f"------------------------------------------------------------")
    print(f"4. Boundary Qualification:")
    print(f"   - Random guesses that successfully met tolerance bounds : {successful_spans}")
    print("=" * 60)

# Run the diagnostic using outputs from your baseline cell
analyze_random_baseline(baseline_gold_data, baseline_pred_data)

      TASK 2 RANDOM BASELINE DIAGNOSTIC ANALYSIS
Total Evaluation Instances : 640
------------------------------------------------------------
1. Routing Distribution:
   - Routed to 'not_propaganda' (-1, -1) : 306 (47.81%)
   - Routed to Active Span Guess        : 334 (52.19%)
------------------------------------------------------------
2. Subsample Composition ('not_propaganda' route):
   - Of those guessed as background, actually gold background : 53.92%
   - Of those guessed as background, actually missed targets   : 46.08%
------------------------------------------------------------
3. Subsample Composition (Active Span route):
   - Of those guessed as spans, held a valid propaganda target : 50.30%
------------------------------------------------------------
4. Boundary Qualification:
   - Random guesses that successfully met tolerance bounds : 11


This means that the capcity for correct random classification guess (similar to task 1) was only 11 instances. Of which all, in this seeded run failed

#### Hyperparameter Sweep

In [191]:
# ==========================================
# HYPERPARAMETER SWEEP FOR VARIATION 2 (10% MODULO DEV SPLIT)
# ==========================================
# Customer Sweep Script. Doesn't use train_deberta_tagger but adapts the code for sweep accross configs
# Swept at 5 epochs


import os
import csv
import torch

os.makedirs("./param_sweep", exist_ok=True)
set_seed(SEED)

TRAIN_PATH = "../data/propaganda_train.tsv"
SWEEP_EPOCHS = 5

# Minimal Search Space Configurations
sweep_configs = [
    {
        "run_name": "Run_1_Conservative",
        "backbone_lr": 1e-5,
        "heads_lr": 5e-4,
        "batch_size": 16,
    },
    {
        "run_name": "Run_2_Moderate",
        "backbone_lr": 2e-5,
        "heads_lr": 1e-3,
        "batch_size": 16,
    },
    {
        "run_name": "Run_3_Aggressive",
        "backbone_lr": 5e-5,
        "heads_lr": 2e-3,
        "batch_size": 32,
    },
]

pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
executor = Task2Executor(pipeline)

global_best_dev_loss = float('inf')
best_run_config = None
sweep_history = []

print("=" * 80)
print(f"       STARTING TASK 2 VARIATION 2 SWEEP ({SWEEP_EPOCHS} EPOCHS | 10% MODULO DEV SPLIT)")
print("=" * 80)

for run_id, config in enumerate(sweep_configs, start=1):
    print(f"\n==================================================")
    print(f" SWEEP RUN {run_id}/{len(sweep_configs)} | {config['run_name']}")
    print(f" Backbone LR: {config['backbone_lr']} | Heads LR: {config['heads_lr']} | Batch Size: {config['batch_size']}")
    print(f"==================================================")
    
    set_seed(SEED)
    model = DebertaCRFTagger(num_tags=17, mode="var2")
    
    optimizer = torch.optim.AdamW([
        {'params': model.deberta.parameters(), 'lr': config['backbone_lr']},
        {'params': model.hidden2tag.parameters(), 'lr': config['heads_lr']},
        {'params': model.crf.parameters(), 'lr': config['heads_lr']}
    ])
    
    accum_steps = config['batch_size']
    epoch_logs = []
    
    # Custom Model Train, does not use train_deberta_tagger
    for epoch in range(1, SWEEP_EPOCHS + 1):
        print("EPOCH:", epoch)
        running_train_loss, train_samples = 0.0, 0
        running_dev_loss, dev_samples = 0.0, 0
        
        with open(TRAIN_PATH, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            optimizer.zero_grad()
            
            for row_idx, row in enumerate(reader, start=1):
                if row_idx % 100 == 0:
                    print(f"Processed {row_idx} instances...")

                label = row['label']
                clean_text, c_start, c_end, _ = pipeline.parse_span_and_clean(row['tagged_in_context'], label)
                input_ids, att_mask, tags = pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode="var2")
                
                # --- 10% MODULO INTERNAL DEV SPLIT ---
                if row_idx % 10 == 0:
                    model.eval()
                    with torch.no_grad():
                        loss = model(input_ids, att_mask, tags)
                        running_dev_loss += loss.item()
                        dev_samples += 1
                else:
                    model.train()
                    loss = model(input_ids, att_mask, tags)
                    scaled_loss = loss / accum_steps
                    scaled_loss.backward()
                    running_train_loss += loss.item()
                    train_samples += 1
                    
                    if row_idx % accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                        optimizer.zero_grad()
            
            # Flush final batch gradients
            if train_samples % accum_steps != 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                
        avg_train_loss = running_train_loss / train_samples if train_samples > 0 else 0.0
        avg_dev_loss = running_dev_loss / dev_samples if dev_samples > 0 else 0.0
        
        print(f"Epoch {epoch}/{SWEEP_EPOCHS} | Avg Train Loss: {avg_train_loss:.4f} | Avg Internal Dev Loss: {avg_dev_loss:.4f}")
        
        epoch_logs.append({
            "epoch": epoch,
            "train_loss": round(avg_train_loss, 4),
            "dev_loss": round(avg_dev_loss, 4)
        })
    
    print("FINISHED EPOCHS")

    # Best dev loss achieved in this run across the 5 epochs
    min_run_dev_loss = min(log["dev_loss"] for log in epoch_logs)
    
    if min_run_dev_loss < global_best_dev_loss:
        global_best_dev_loss = min_run_dev_loss
        best_run_config = config
        print(f"🔥 NEW BEST CONFIGURATION! Internal Dev Loss: {global_best_dev_loss:.4f}")
        
    sweep_history.append({
        "run_id": run_id,
        "config": config,
        "epoch_logs": epoch_logs
    })

print("FINISHED CONFIGS")

print("\n" + "=" * 80)
print(f"SWEEP COMPLETE!")
print(f"Best Configuration   : {best_run_config['run_name']}")
print(f"Lowest Dev NLL Loss  : {global_best_dev_loss:.4f}")
print("=" * 80)

# ==========================================
# EXPORT RESULTS TO CSV
# ==========================================
csv_filename = "./param_sweep/sweep_results_var2.csv"
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    header = ["run_id", "run_name", "backbone_lr", "heads_lr", "batch_size"]
    for ep in range(1, SWEEP_EPOCHS + 1):
        header.extend([f"ep_{ep}_train_loss", f"ep_{ep}_dev_loss"])
    writer.writerow(header)
    
    for item in sweep_history:
        r_id = item["run_id"]
        cfg = item["config"]
        logs = item["epoch_logs"]
        
        row = [r_id, cfg["run_name"], cfg["backbone_lr"], cfg["heads_lr"], cfg["batch_size"]]
        for ep_log in logs:
            row.extend([ep_log["train_loss"], ep_log["dev_loss"]])
        writer.writerow(row)

print(f"\nSuccessfully saved sweep details to '{csv_filename}'.")

Global seed set to 142
       STARTING TASK 2 VARIATION 2 SWEEP (5 EPOCHS | 10% MODULO DEV SPLIT)

 SWEEP RUN 1/3 | Run_1_Conservative
 Backbone LR: 1e-05 | Heads LR: 0.0005 | Batch Size: 16
Global seed set to 142


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 48070.86it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

384
EPOCH: 1
Processed 100 instances...
Processed 200 instances...
Processed 300 instances...
Processed 400 instances...
Processed 500 instances...
Processed 600 instances...
Processed 700 instances...
Processed 800 instances...
Processed 900 instances...
Processed 1000 instances...
Processed 1100 instances...
Processed 1200 instances...
Processed 1300 instances...
Processed 1400 instances...
Processed 1500 instances...
Processed 1600 instances...
Processed 1700 instances...
Processed 1800 instances...
Processed 1900 instances...
Processed 2000 instances...
Processed 2100 instances...
Processed 2200 instances...
Processed 2300 instances...
Processed 2400 instances...
Processed 2500 instances...
Epoch 1/5 | Avg Train Loss: 13.0195 | Avg Internal Dev Loss: 13.5711
EPOCH: 2
Processed 100 instances...
Processed 200 instances...
Processed 300 instances...
Processed 400 instances...
Processed 500 instances...
Processed 600 instances...
Processed 700 instances...
Processed 800 instances...
Pr

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 41198.14it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

384
EPOCH: 1
Processed 100 instances...
Processed 200 instances...
Processed 300 instances...
Processed 400 instances...
Processed 500 instances...
Processed 600 instances...
Processed 700 instances...
Processed 800 instances...
Processed 900 instances...
Processed 1000 instances...
Processed 1100 instances...
Processed 1200 instances...
Processed 1300 instances...
Processed 1400 instances...
Processed 1500 instances...
Processed 1600 instances...
Processed 1700 instances...
Processed 1800 instances...
Processed 1900 instances...
Processed 2000 instances...
Processed 2100 instances...
Processed 2200 instances...
Processed 2300 instances...
Processed 2400 instances...
Processed 2500 instances...
Epoch 1/5 | Avg Train Loss: 9.7314 | Avg Internal Dev Loss: 9.9557
EPOCH: 2
Processed 100 instances...
Processed 200 instances...
Processed 300 instances...
Processed 400 instances...
Processed 500 instances...
Processed 600 instances...
Processed 700 instances...
Processed 800 instances...
Proc

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 38687.79it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

384
EPOCH: 1
Processed 100 instances...
Processed 200 instances...
Processed 300 instances...
Processed 400 instances...
Processed 500 instances...
Processed 600 instances...
Processed 700 instances...
Processed 800 instances...
Processed 900 instances...
Processed 1000 instances...
Processed 1100 instances...
Processed 1200 instances...
Processed 1300 instances...
Processed 1400 instances...
Processed 1500 instances...
Processed 1600 instances...
Processed 1700 instances...
Processed 1800 instances...
Processed 1900 instances...
Processed 2000 instances...
Processed 2100 instances...
Processed 2200 instances...
Processed 2300 instances...
Processed 2400 instances...
Processed 2500 instances...
Epoch 1/5 | Avg Train Loss: 9.9138 | Avg Internal Dev Loss: 10.1360
EPOCH: 2
Processed 100 instances...
Processed 200 instances...
Processed 300 instances...
Processed 400 instances...
Processed 500 instances...
Processed 600 instances...
Processed 700 instances...
Processed 800 instances...
Pro


**Best Run: Conservative**
- Backbone LR: 1e-5
- Heads LR:    5e-4
- Batch Size:  16

*Values updates within class structure*

#### Variation 2 (17-Class Joint Tagger)

In [213]:
# ==========================================
# CELL 8: Variation 2 (17-Class Joint Tagger)
# ==========================================

set_seed(SEED)

# 1. File Paths & Winning Conservative Hyperparameters
TRAIN_PATH = '../data/propaganda_train.tsv'
VAL_PATH = '../data/propaganda_val.tsv'
VAR2_MODEL_SAVE = './final_models/var2_deberta_crf_joint.pt'

EPOCHS = 10             
BATCH_SIZE = 16          # Simulated batch size via gradient accumulation
BACKBONE_LR = 1e-5       # Conservative Backbone LR (0.00001)
HEADS_LR = 5e-4          # Conservative Heads LR (0.0005)

print("=" * 60)
print("     STARTING TASK 2 VARIATION 2 (CONSERVATIVE RUN)")
print("=" * 60)

t2_pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
t2_executor = Task2Executor(pipeline=t2_pipeline)

# 2. Train using the winning parameters explicitly passed from the script
var2_model = t2_executor.train_deberta_tagger(
    train_path=TRAIN_PATH,
    mode="var2",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    backbone_lr=BACKBONE_LR,
    heads_lr=HEADS_LR
)

# 3. Save Model Weights
import os
os.makedirs('./final_models', exist_ok=True)
torch.save(var2_model.state_dict(), VAR2_MODEL_SAVE)
print(f"\n--> Variation 2 Joint Model saved successfully to {VAR2_MODEL_SAVE}\n")

# 4. Evaluate Model on Validation Set
print("=" * 60)
print("     EVALUATING VARIATION 2 ON VALIDATION SET")
print("=" * 60)

var2_metrics, var2_class_report = t2_executor.evaluate_tagger(
    model=var2_model,
    test_path=VAL_PATH,
    mode="var2"
)

# 5. Report Primary Benchmark Metrics
print(f"\n==========================================")
print(f" VARIATION 2 EVALUATION REPORT (VALIDATION)")
print(f"==========================================")
print(f" Macro-F1 Score:  {var2_metrics['Macro-F1']:.4f}")
print(f" Macro Precision: {var2_metrics['Precision']:.4f}")
print(f" Macro Recall:    {var2_metrics['Recall']:.4f}")
print(f"------------------------------------------")
print(var2_class_report)
print(f"==========================================\n")

Global seed set to 142
     STARTING TASK 2 VARIATION 2 (CONSERVATIVE RUN)

--- Training DeBERTa-CRF (VAR2) [Batch Size: 16] ---


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 49292.03it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

384
  Processed 100 instances...
  Processed 200 instances...
  Processed 300 instances...
  Processed 400 instances...
  Processed 500 instances...
  Processed 600 instances...
  Processed 700 instances...
  Processed 800 instances...
  Processed 900 instances...
  Processed 1000 instances...
  Processed 1100 instances...
  Processed 1200 instances...
  Processed 1300 instances...
  Processed 1400 instances...
  Processed 1500 instances...
  Processed 1600 instances...
  Processed 1700 instances...
  Processed 1800 instances...
  Processed 1900 instances...
  Processed 2000 instances...
  Processed 2100 instances...
  Processed 2200 instances...
  Processed 2300 instances...
  Processed 2400 instances...
  Processed 2500 instances...
Epoch 1 | Total CRF NLL Loss: 30155.8526
  Processed 100 instances...
  Processed 200 instances...
  Processed 300 instances...
  Processed 400 instances...
  Processed 500 instances...
  Processed 600 instances...
  Processed 700 instances...
  Processed

In [262]:
# ==========================================
# AD-HOC EVALUATION: VARIATION 2
# ==========================================

set_seed(SEED)

VAL_PATH = '../data/propaganda_val.tsv'
VAR2_MODEL_SAVE = './final_models/var2_deberta_crf_joint.pt'

print("=" * 60)
print("     LOADING & EVALUATING VARIATION 2 AD-HOC")
print("=" * 60)

# 1. Initialize pipeline and executor
t2_pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
t2_executor = Task2Executor(pipeline=t2_pipeline)

# 2. Re-instantiate the model architecture for Variation 2 and move to device
loaded_var2_model = DebertaCRFTagger(mode="var2").to(device)

# 3. Load the saved weights from disk
loaded_var2_model.load_state_dict(torch.load(VAR2_MODEL_SAVE, map_location=device, weights_only=True))

# 4. Run evaluation using the executor
var2_metrics, var2_class_report = t2_executor.evaluate_tagger(
    model=loaded_var2_model,
    test_path=VAL_PATH,
    mode="var2"
)

# 5. Report Primary Benchmark Metrics
print(f"\n==========================================")
print(f" VARIATION 2 EVALUATION REPORT (VALIDATION)")
print(f"==========================================")
print(f" Macro-F1 Score:  {var2_metrics['Macro-F1']:.4f}")
print(f" Macro Precision: {var2_metrics['Precision']:.4f}")
print(f" Macro Recall:    {var2_metrics['Recall']:.4f}")
print(f"------------------------------------------")
print(var2_class_report)
print(f"==========================================\n")

Global seed set to 142
     LOADING & EVALUATING VARIATION 2 AD-HOC


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 51604.56it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384

 VARIATION 2 EVALUATION REPORT (VALIDATION)
 Macro-F1 Score:  0.2034
 Macro Precision: 0.2914
 Macro Recall:    0.1698
------------------------------------------
                           precision    recall  f1-score   support

              flag_waving       0.29      0.16      0.20        45
 appeal_to_fear_prejudice       0.32      0.19      0.24        43
causal_oversimplification       0.42      0.31      0.36        35
                    doubt       0.32      0.23      0.27        43
          loaded_language       0.10      0.10      0.10        39
    name_calling,labeling       0.50      0.12      0.19        34
               repetition       0.20      0.05      0.08        40
exaggeration,minimisation       0.18      0.20      0.19        30

                micro avg       0.26      0.17      0.20       309
                macro avg       0.29      0.17      0.20       309
             weighted avg       0.29      0.17      0.20       309




#### Hyper Sweep

In [281]:
# =====================================================================
# VARIATION 1: RANDOM SEARCH HYPERPARAMETER SWEEP (DUAL-HEAD ARCHITECTURE)
# =====================================================================

import csv
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

# 1. Sweep Configuration & Fixed Parameters
SEED = 42
set_seed(SEED)

TRAIN_PATH = '../data/propaganda_train.tsv'
SWEEP_EPOCHS = 3          # Efficient 3-5 epoch budget
BACKBONE_LR = 1e-5        # Fixed conservative rate for DeBERTa backbone
NUM_TRIALS = 6            # Number of random combinations to explore

# Define the Random Search Space
PARAM_SPACE = {
    "boundary_lr": [1e-4, 3e-4, 5e-4, 8e-4],
    "technique_lr": [3e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "technique_dropout": [0.1, 0.2, 0.3, 0.4]
}

print("=" * 80)
print("     STARTING VARIATION 1 RANDOM HYPERPARAMETER SWEEP")
print("=" * 80)

# 2. Setup Data Pipeline & 10% Modulo Dev Split
t2_pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
t2_executor = Task2Executor(pipeline=t2_pipeline)

# Load full training rows to create indices for modulo split
all_rows = []
with open(TRAIN_PATH, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file, delimiter="\t", quoting=csv.QUOTE_NONE)
    for row in reader:
        all_rows.append(row)

# 10% Modulo Dev Split (idx % 10 == 0 -> dev, else -> train)
train_indices = [i for i in range(len(all_rows)) if i % 10 != 0]
dev_indices   = [i for i in range(len(all_rows)) if i % 10 == 0]

print(f"Data Split Loaded -> Train Rows: {len(train_indices)} | Modulo Dev Rows: {len(dev_indices)}\n")

# 3. Sweep Tracking Variables
best_macro_f1 = 0.0
best_config = {}
sweep_results = []

# 4. Execute Random Search Trials
for trial in range(1, NUM_TRIALS + 1):
    # Sample a random hyperparameter combination
    b_lr = random.choice(PARAM_SPACE["boundary_lr"])
    t_lr = random.choice(PARAM_SPACE["technique_lr"])
    t_drop = random.choice(PARAM_SPACE["technique_dropout"])
    
    print(f"\n--- [TRIAL {trial}/{NUM_TRIALS}] b_lr={b_lr} | t_lr={t_lr} | t_drop={t_drop} ---")
    
    # Instantiate Model & Inject Custom Dropout into Technique Head MLP
    model = DebertaCRFTagger(mode="var1").to(device)
    
    # Dynamically update dropout layer inside technique_head (MLP: Linear -> Dropout -> ReLU -> Linear)
    for module in model.technique_head.modules():
        if isinstance(module, nn.Dropout):
            module.p = t_drop
            
    # Configure Decoupled Parameter Groups
    boundary_params = list(model.boundary_head.parameters()) + list(model.crf.parameters())
    technique_params = list(model.technique_head.parameters())
    
    optimizer = torch.optim.AdamW([
        {'params': model.deberta.parameters(), 'lr': BACKBONE_LR},
        {'params': boundary_params,            'lr': b_lr},
        {'params': technique_params,           'lr': t_lr}
    ])
    
    # Simple training loop for SWEEP_EPOCHS
    model.train()
    for epoch in range(1, SWEEP_EPOCHS + 1):
        total_loss = 0.0
        # Stream train split rows
        for idx in train_indices:
            row = all_rows[idx]
            label = row["label"]
            clean_text, c_start, c_end, _ = t2_pipeline.parse_span_and_clean(row["tagged_in_context"], label)
            input_ids, att_mask, tags = t2_pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode="var1")
            
            input_ids, att_mask, tags = input_ids.to(device), att_mask.to(device), tags.to(device)
            
            optimizer.zero_grad()
            loss = model(input_ids, att_mask, tags=tags)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"   Epoch {epoch}/{SWEEP_EPOCHS} | Train Loss: {total_loss / len(train_indices):.4f}")
        
    # Evaluate on the 10% Modulo Dev Split
    model.eval()
    gold_data, pred_data = [], []
    with torch.no_grad():
        for idx in dev_indices:
            row = all_rows[idx]
            label = row["label"]
            clean_text, c_start, c_end, _ = t2_pipeline.parse_span_and_clean(row["tagged_in_context"], label)
            input_ids, att_mask, tags = t2_pipeline.align_bio_tags(clean_text, c_start, c_end, label, mode="var1")
            g_start, g_end, _ = t2_executor.extract_viterbi_span(tags[0].tolist(), mode="var1")
            
            viterbi_path = model(input_ids.to(device), att_mask.to(device))[0]
            p_start, p_end, p_tech = t2_executor.extract_viterbi_span(viterbi_path, mode="var1")
            
            gold_data.append({"span": (g_start, g_end), "technique": label})
            pred_data.append({"span": (p_start, p_end), "technique": p_tech})
            
    eval_metrics, _ = Task2Evaluator.evaluate_predictions(gold_data, pred_data)
    current_f1 = eval_metrics["Macro-F1"]
    
    print(f"   --> Trial {trial} Dev Macro-F1: {current_f1:.4f}")
    sweep_results.append({
        "trial": trial, "b_lr": b_lr, "t_lr": t_lr, 
        "t_drop": t_drop, "dev_macro_f1": current_f1
    })
    
    # Save best performing weights
    if current_f1 > best_macro_f1:
        best_macro_f1 = current_f1
        best_config = {"b_lr": b_lr, "t_lr": t_lr, "t_drop": t_drop}
        torch.save(model.state_dict(), "./final_models/var1_best_sweep_checkpoint.pt")
        print(f"   🏆 New Best Configuration Found! SavedCheckpoint.")

# 5. Final Report
print("\n" + "=" * 80)
print("     VARIATION 1 HYPERPARAMETER SWEEP SUMMARY")
print("=" * 80)
for res in sweep_results:
    print(f"Trial {res['trial']:02d} | b_lr={res['b_lr']} | t_lr={res['t_lr']} | t_drop={res['t_drop']} | Dev Macro-F1: {res['dev_macro_f1']:.4f}")
print("-" * 80)
print(f"Best Configuration : {best_config}")
print(f"Best Dev Macro-F1  : {best_macro_f1:.4f}")
print("=" * 80)

Global seed set to 42
     STARTING VARIATION 1 RANDOM HYPERPARAMETER SWEEP
Data Split Loaded -> Train Rows: 2304 | Modulo Dev Rows: 256


--- [TRIAL 1/6] b_lr=0.0001 | t_lr=0.0003 | t_drop=0.3 ---


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 39936.15it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
   Epoch 1/3 | Train Loss: 13.2861
   Epoch 2/3 | Train Loss: 7.9413
   Epoch 3/3 | Train Loss: 5.7228
   --> Trial 1 Dev Macro-F1: 0.0446
   🏆 New Best Configuration Found! SavedCheckpoint.

--- [TRIAL 2/6] b_lr=0.0003 | t_lr=0.0005 | t_drop=0.2 ---


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 43882.28it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
   Epoch 1/3 | Train Loss: 12.3096
   Epoch 2/3 | Train Loss: 6.4597
   Epoch 3/3 | Train Loss: 4.1071
   --> Trial 2 Dev Macro-F1: 0.0435

--- [TRIAL 3/6] b_lr=0.0001 | t_lr=0.003 | t_drop=0.1 ---


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 34435.14it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
   Epoch 1/3 | Train Loss: 12.7678
   Epoch 2/3 | Train Loss: 8.0638


KeyboardInterrupt: 

#### Variation 1 (Dual-Head Additive Tagger)

In [266]:
# ==========================================
# CELL 9: Variation 1 (Dual-Head Additive Tagger)
# ==========================================

set_seed(SEED)

# 1. File Paths & Winning Conservative Hyperparameters
TRAIN_PATH = '../data/propaganda_train.tsv'
VAL_PATH = '../data/propaganda_val.tsv'
VAR1_MODEL_SAVE = './final_models/var1_deberta_crf_dual.pt'

EPOCHS = 10             
BATCH_SIZE = 16          
BACKBONE_LR = 1e-5     
HEADS_LR = 5e-4 

print("=" * 60)
print("     STARTING TASK 2 VARIATION 1 (DUAL-HEAD ADDITIVE RUN)")
print("=" * 60)

t2_pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
t2_executor = Task2Executor(pipeline=t2_pipeline)

# 2. Train using mode="var1"
var1_model = t2_executor.train_deberta_tagger(
    train_path=TRAIN_PATH,
    mode="var1",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    backbone_lr=BACKBONE_LR,
    heads_lr=HEADS_LR
)

# 3. Save Model Weights
import os
os.makedirs('./final_models', exist_ok=True)
torch.save(var1_model.state_dict(), VAR1_MODEL_SAVE)
print(f"\n--> Variation 1 Dual-Head Model saved successfully to {VAR1_MODEL_SAVE}\n")

Global seed set to 142
     STARTING TASK 2 VARIATION 1 (DUAL-HEAD ADDITIVE RUN)

--- Training DeBERTa-CRF (VAR1) [Batch Size: 16] ---


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 48608.26it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
  Processed 100 instances...
  Processed 200 instances...
  Processed 300 instances...
  Processed 400 instances...
  Processed 500 instances...
  Processed 600 instances...
  Processed 700 instances...
  Processed 800 instances...
  Processed 900 instances...
  Processed 1000 instances...
  Processed 1100 instances...
  Processed 1200 instances...
  Processed 1300 instances...
  Processed 1400 instances...
  Processed 1500 instances...
  Processed 1600 instances...
  Processed 1700 instances...
  Processed 1800 instances...
  Processed 1900 instances...
  Processed 2000 instances...
  Processed 2100 instances...
  Processed 2200 instances...
  Processed 2300 instances...
  Processed 2400 instances...
  Processed 2500 instances...
Epoch 1 | Total CRF NLL Loss: 25979.0038
  Processed 100 instances...
  Processed 200 instances...
  Processed 300 instances...
  Processed 400 instances...
  Processed 500 instances...
  Processed 600 instances...
  Processed 700 ins

In [272]:
# ==========================================
# AD-HOC EVALUATION: VARIATION 1
# ==========================================

set_seed(SEED)

VAL_PATH = '../data/propaganda_train.tsv'
VAR1_MODEL_SAVE = './final_models/var1_deberta_crf_dual.pt'

print("=" * 60)
print("     LOADING & EVALUATING VARIATION 1 AD-HOC")
print("=" * 60)

# 1. Initialize pipeline and executor
t2_pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
t2_executor = Task2Executor(pipeline=t2_pipeline)

# 2. Re-instantiate the model architecture for Variation 2 and move to device
loaded_var1_model = DebertaCRFTagger(mode="var1").to(device)

# 3. Load the saved weights from disk
loaded_var1_model.load_state_dict(torch.load(VAR1_MODEL_SAVE, map_location=device, weights_only=True))

# 4. Run evaluation using the executor
var1_metrics, var1_class_report = t2_executor.evaluate_tagger(
    model=loaded_var1_model,
    test_path=VAL_PATH,
    mode="var1"
)

# 5. Report Primary Benchmark Metrics
print(f"\n==========================================")
print(f" VARIATION 1 EVALUATION REPORT (VALIDATION)")
print(f"==========================================")
print(f" Macro-F1 Score:  {var1_metrics['Macro-F1']:.4f}")
print(f" Macro Precision: {var1_metrics['Precision']:.4f}")
print(f" Macro Recall:    {var1_metrics['Recall']:.4f}")
print(f"------------------------------------------")
print(var1_class_report)
print(f"==========================================\n")

Global seed set to 142
     LOADING & EVALUATING VARIATION 1 AD-HOC


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 42922.90it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384

 VARIATION 1 EVALUATION REPORT (VALIDATION)
 Macro-F1 Score:  0.0796
 Macro Precision: 0.1100
 Macro Recall:    0.0660
------------------------------------------
                           precision    recall  f1-score   support

              flag_waving       0.07      0.06      0.06       155
 appeal_to_fear_prejudice       0.00      0.00      0.00       157
causal_oversimplification       0.00      0.00      0.00       165
                    doubt       0.00      0.00      0.00       157
          loaded_language       0.25      0.23      0.24       161
    name_calling,labeling       0.09      0.05      0.07       166
               repetition       0.43      0.16      0.24       160
exaggeration,minimisation       0.04      0.02      0.03       170

                micro avg       0.09      0.07      0.08      1291
                macro avg       0.11      0.07      0.08      1291
             weighted avg       0.11      0.07      0.08      1291




#### Analysis

##### Var2

In [229]:
# ==========================================
# VARIATION 2 TRAINING SET PREDICTION INSPECTOR
# ==========================================

import csv
import torch

def inspect_var2_training_predictions(
    model_path="./final_models/var2_deberta_crf_joint.pt",
    train_path="../data/propaganda_train.tsv",
    num_examples=25
):
    """
    Loads the Variation 2 joint 17-class tagger, streams the training dataset,
    and prints a side-by-side comparison of gold vs. predicted spans and labels[cite: 2].
    """
    # 1. Initialize data pipeline and execution engine[cite: 2]
    pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
    executor = Task2Executor(pipeline)
    
    # 2. Instantiate Tagger in mode="var2" (17-Class Joint CRF)[cite: 2]
    model = DebertaCRFTagger(mode="var2").to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    
    print("=" * 80)
    print(f"  VARIATION 2 TRAINING SET INSPECTION | SHOWING TOP {num_examples} ACTIVE EXAMPLES")
    print("=" * 80)
    
    printed_count = 0
    
    with open(train_path, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file, delimiter="\t", quoting=csv.QUOTE_NONE)
        
        for idx, row in enumerate(reader, start=1):
            if printed_count >= num_examples:
                break
                
            label = row["label"]
            clean_text, c_start, c_end, _ = pipeline.parse_span_and_clean(
                row["tagged_in_context"], label
            )
            
            # Generate 17-class target IDs for span extraction/alignment[cite: 2]
            input_ids, att_mask, tags = pipeline.align_bio_tags(
                clean_text, c_start, c_end, label, mode="var2"
            )
            
            # Gold Span Recovery[cite: 2]
            gold_tag_ids = tags[0].tolist()
            g_start, g_end, _ = executor.extract_viterbi_span(gold_tag_ids, mode="var2")
            gold_snippet = (
                clean_text[c_start:c_end] 
                if label != "not_propaganda" 
                else "N/A (not_propaganda)"
            )
            
            # Move tensors to active M1 GPU / CPU device[cite: 2]
            input_ids = input_ids.to(device)
            att_mask = att_mask.to(device)

            # Model Inference Pass -> CRF Viterbi Decoding[cite: 2]
            with torch.no_grad():
                viterbi_path = model(input_ids, att_mask)[0]
                
            p_start, p_end, p_tech = executor.extract_viterbi_span(viterbi_path, mode="var2")
            
            # Skip True Negatives to keep diagnostic output scannable[cite: 2]
            if label == "not_propaganda" and p_tech == "not_propaganda":
                continue
            
            printed_count += 1
            
            # Map subword token indices back to string character offsets[cite: 2]
            if p_start != -1 and p_end != -1:
                offsets = pipeline.tokenizer(
                    clean_text, return_offsets_mapping=True
                )["offset_mapping"]
                
                if p_end < len(offsets):
                    pred_char_start = offsets[p_start][0]
                    pred_char_end = offsets[p_end][1]
                    pred_snippet = clean_text[pred_char_start:pred_char_end]
                else:
                    pred_snippet = "Span boundary out of range"
            else:
                pred_snippet = "N/A (No span detected)"

            # Print Structured Diagnostic Block[cite: 2]
            print(f"--- Example #{printed_count} (Dataset Row #{idx}) ---")
            print(f"Full Text : \"{clean_text}\"")
            print(f"GOLD      : [{label}] -> \"{gold_snippet}\" (Tokens: {g_start} to {g_end})")
            print(f"PREDICTED : [{p_tech}] -> \"{pred_snippet}\" (Tokens: {p_start} to {p_end})")
            print("-" * 80)

# Execute inspection over the training set[cite: 2]
inspect_var2_training_predictions(
    model_path="./final_models/var2_deberta_crf_joint.pt",
    train_path="../data/propaganda_train.tsv",
    num_examples=25
)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 46564.18it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
  VARIATION 2 TRAINING SET INSPECTION | SHOWING TOP 25 ACTIVE EXAMPLES
--- Example #1 (Dataset Row #3) ---
Full Text : ""The Obama administration misled the  American people  and Congress because they were desperate to get a deal with Iran," said Sen."
GOLD      : [flag_waving] -> " American people " (Tokens: 7 to 8)
PREDICTED : [flag_waving] -> " American people" (Tokens: 7 to 8)
--------------------------------------------------------------------------------
--- Example #2 (Dataset Row #6) ---
Full Text : "Hitler  annihilated  400,000 Germans who were handicapped or suffered from chronic diseases."
GOLD      : [loaded_language] -> " annihilated " (Tokens: 2 to 2)
PREDICTED : [loaded_language] -> " annihilated" (Tokens: 2 to 2)
--------------------------------------------------------------------------------
--- Example #3 (Dataset Row #9) ---
Full Text : "As noted above, at this point literally every piece of  so-called evidence  put forth by authorities and t

##### Var1

In [271]:
# ==========================================
# VARIATION 1 TRAINING SET PREDICTION INSPECTOR
# ==========================================

import csv
import torch

def inspect_var1_training_predictions(
    model_path="./final_models/var1_deberta_crf_dual.pt",
    train_path="../data/propaganda_train.tsv",
    num_examples=25
):
    """
    Loads the Variation 1 dual-head additive tagger, streams the training dataset,
    and prints a side-by-side comparison of gold vs. predicted spans and labels[cite: 2].
    """
    # 1. Initialize data pipeline and execution engine[cite: 2]
    pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
    executor = Task2Executor(pipeline)
    
    # 2. Instantiate Tagger in mode="var1" (Dual-Head Additive CRF)[cite: 2]
    model = DebertaCRFTagger(mode="var1").to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    
    print("=" * 80)
    print(f"  VARIATION 1 TRAINING SET INSPECTION | SHOWING TOP {num_examples} ACTIVE EXAMPLES")
    print("=" * 80)
    
    printed_count = 0
    
    with open(train_path, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file, delimiter="\t", quoting=csv.QUOTE_NONE)
        
        for idx, row in enumerate(reader, start=1):
            if printed_count >= num_examples:
                break
                
            label = row["label"]
            clean_text, c_start, c_end, _ = pipeline.parse_span_and_clean(
                row["tagged_in_context"], label
            )
            
            # Generate 17-class target IDs because Var1 compiles into 17-class CRF logits[cite: 2]
            input_ids, att_mask, tags = pipeline.align_bio_tags(
                clean_text, c_start, c_end, label, mode="var2"
            )
            
            # Gold Span Recovery using 17-class schema[cite: 2]
            gold_tag_ids = tags[0].tolist()
            g_start, g_end, _ = executor.extract_viterbi_span(gold_tag_ids, mode="var2")
            gold_snippet = (
                clean_text[c_start:c_end] 
                if label != "not_propaganda" 
                else "N/A (not_propaganda)"
            )
            
            # Move tensors to active GPU / CPU device[cite: 2]
            input_ids = input_ids.to(device)
            att_mask = att_mask.to(device)

            # Model Inference Pass -> Dual-Head Compilation -> CRF Viterbi Decoding[cite: 2]
            with torch.no_grad():
                viterbi_path = model(input_ids, att_mask)[0]
                
            p_start, p_end, p_tech = executor.extract_viterbi_span(viterbi_path, mode="var2")
            
            # Skip True Negatives to keep diagnostic output readable[cite: 2]
            if label == "not_propaganda" and p_tech == "not_propaganda":
                continue
            
            printed_count += 1
            
            # Map subword token indices back to string character offsets[cite: 2]
            if p_start != -1 and p_end != -1:
                offsets = pipeline.tokenizer(
                    clean_text, return_offsets_mapping=True
                )["offset_mapping"]
                
                if p_end < len(offsets):
                    pred_char_start = offsets[p_start][0]
                    pred_char_end = offsets[p_end][1]
                    pred_snippet = clean_text[pred_char_start:pred_char_end]
                else:
                    pred_snippet = "Span boundary out of range"
            else:
                pred_snippet = "N/A (No span detected)"

            # Print Structured Diagnostic Block[cite: 2]
            print(f"--- Example #{printed_count} (Dataset Row #{idx}) ---")
            print(f"Full Text : \"{clean_text}\"")
            print(f"GOLD      : [{label}] -> \"{gold_snippet}\" (Tokens: {g_start} to {g_end})")
            print(f"PREDICTED LABEL : [{p_tech}]")
            print(f"PREDICTED SPAN : '{pred_snippet}' (Tokens: {p_start} to {p_end})")
            print("-" * 80)

# Execute inspection over the training set for Variation 1[cite: 2]
inspect_var1_training_predictions(
    model_path="./final_models/var1_deberta_crf_dual.pt",
    train_path="../data/propaganda_val.tsv",
    num_examples=25
)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 46564.18it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
  VARIATION 1 TRAINING SET INSPECTION | SHOWING TOP 25 ACTIVE EXAMPLES
--- Example #1 (Dataset Row #2) ---
Full Text : "Mostly because  the country would not last long without an outside high IQ elite to run the country ."
GOLD      : [causal_oversimplification] -> " the country would not last long without an outside high IQ elite to run the country " (Tokens: 3 to 18)
PREDICTED LABEL : [flag_waving]
PREDICTED SPAN : 'Mostly because  the country would not last long without an outside high IQ elite to run the country' (Tokens: 1 to 18)
--------------------------------------------------------------------------------
--- Example #2 (Dataset Row #3) ---
Full Text : "Lyndon Johnson  gets Earl Warren and Sen. Richard Russel to join the Warren Commission by telling them that the assassination could lead to World War III ."
GOLD      : [appeal_to_fear_prejudice] -> " gets Earl Warren and Sen. Richard Russel to join the Warren Commission by telling them that the assassi

In [278]:
# ==========================================
# COMPREHENSIVE VARIATION 1 DIAGNOSTIC INSPECTOR
# ==========================================

import csv
import torch

def run_comprehensive_diagnostics(
    model_path="./final_models/var1_deberta_crf_dual.pt",
    dataset_path="../data/propaganda_val.tsv",
    mode="var1"
):
    """
    Streams through a dataset, performs Variation 1 inference, evaluates tolerance 
    bounds and technique matching, and prints a comprehensive diagnostic table.
    """
    pipeline = Task2DataPipeline(model_checkpoint="microsoft/deberta-v3-xsmall")
    executor = Task2Executor(pipeline)
    
    # Instantiate and load model weights
    model = DebertaCRFTagger(mode=mode).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    
    print("=" * 120)
    print(f"  COMPREHENSIVE DIAGNOSTIC REPORT | DATASET: {dataset_path} | MODE: {mode.upper()}")
    print("=" * 120)
    
    with open(dataset_path, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file, delimiter="\t", quoting=csv.QUOTE_NONE)
        
        for idx, row in enumerate(reader, start=1):
            label = row["label"]
            clean_text, c_start, c_end, _ = pipeline.parse_span_and_clean(
                row["tagged_in_context"], label
            )
            
            # 1. Gold Extraction
            input_ids, att_mask, tags = pipeline.align_bio_tags(
                clean_text, c_start, c_end, label, mode="var2"
            )
            g_start, g_end, _ = executor.extract_viterbi_span(tags[0].tolist(), mode="var2")
            
            gold_snippet = clean_text[c_start:c_end] if label != "not_propaganda" else "N/A"
            gold_len = (g_end - g_start + 1) if g_start != -1 else 0
            
            # 2. Model Prediction Inference
            input_ids = input_ids.to(device)
            att_mask = att_mask.to(device)
            
            with torch.no_grad():
                viterbi_path = model(input_ids, att_mask)[0]
                
            p_start, p_end, p_tech = executor.extract_viterbi_span(viterbi_path, mode="var2")
            
            # Extract predicted character text via token offset mapping
            offsets = pipeline.tokenizer(clean_text, return_offsets_mapping=True)["offset_mapping"]
            if p_start != -1 and p_end != -1 and p_end < len(offsets):
                pred_char_start = offsets[p_start][0]
                pred_char_end = offsets[p_end][1]
                pred_snippet = clean_text[pred_char_start:pred_char_end]
            else:
                pred_snippet = "No Span Detected"
                
            # 3. Tolerance & Error Analysis Calculations
            if g_start != -1:
                tolerance = Task2Evaluator.get_tolerance(gold_len)
                left_diff = abs(p_start - g_start)
                right_diff = abs(p_end - g_end)
                
                left_failed = "FAIL" if left_diff > tolerance else "PASS"
                right_failed = "FAIL" if right_diff > tolerance else "PASS"
                boundary_status = "QUALIFIED" if (left_diff <= tolerance and right_diff <= tolerance) else "DISQUALIFIED"
            else:
                tolerance = 0
                left_diff, right_diff = 0, 0
                left_failed, right_failed = "N/A", "N/A"
                boundary_status = "N/A (Gold Background)"
                
            tech_matched = "YES" if (label == p_tech) else "NO"

            if (label == 'not_propaganda') and (p_tech == 'not_propaganda'): continue
            else: 
                print(f"Row #{idx} | Full Text: \"{clean_text}\"")
                print(f"  [GOLD]      Technique: {label:<25} | Span Token Index: ({g_start}, {g_end}) | Text: \"{gold_snippet}\"")
                print(f"  [PREDICTED] Technique: {p_tech:<25} | Span Token Index: ({p_start}, {p_end}) | Text: \"{pred_snippet}\"")
                print(f"  [METRICS]   Tolerance Allowed: {tolerance} | Left Bound: {left_failed} (Δ={left_diff}) | Right Bound: {right_failed} (Δ={right_diff})")
                print(f"              Boundary Status: {boundary_status:<12} | Technique Match: {tech_matched}")
                print("-" * 120)

            if (label != 'not_propaganda') and (p_tech == 'not_propaganda'):
                print("FAILURE TO IDENTIFY PROPAGANDA")
                

# Run diagnostics across the validation split
run_comprehensive_diagnostics(
    model_path="./final_models/var1_deberta_crf_dual.pt",
    dataset_path="../data/propaganda_val.tsv",
    mode="var1"
)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 41467.63it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNE

DeBERTa Hidden Size: 384
  COMPREHENSIVE DIAGNOSTIC REPORT | DATASET: ../data/propaganda_val.tsv | MODE: VAR1
Row #2 | Full Text: "Mostly because  the country would not last long without an outside high IQ elite to run the country ."
  [GOLD]      Technique: causal_oversimplification | Span Token Index: (3, 18) | Text: " the country would not last long without an outside high IQ elite to run the country "
  [PREDICTED] Technique: flag_waving               | Span Token Index: (1, 18) | Text: "Mostly because  the country would not last long without an outside high IQ elite to run the country"
  [METRICS]   Tolerance Allowed: 4 | Left Bound: PASS (Δ=2) | Right Bound: PASS (Δ=0)
              Boundary Status: QUALIFIED    | Technique Match: NO
------------------------------------------------------------------------------------------------------------------------
Row #3 | Full Text: "Lyndon Johnson  gets Earl Warren and Sen. Richard Russel to join the Warren Commission by telling them that 

#### Additional Code 1: Subword Max Length

Ensuring instances do not exceed DeBERTas max length capcity. 

In [8]:
import numpy as np
from collections import Counter

def analyze_dataset_token_lengths(pipeline, train_path):
    token_counts = []
    
    with open(train_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
        for row in reader:
            clean_text, _, _, _ = pipeline.parse_span_and_clean(
                row['tagged_in_context'], row['label']
            )
            # Tokenize using your pipeline's DeBERTa tokenizer without truncation
            tokens = pipeline.tokenizer.encode(clean_text, add_special_tokens=True)
            token_counts.append(len(tokens))
            
    token_counts = np.array(token_counts)
    
    print("=" * 50)
    print("      DEBERTA SUBWORD TOKEN LENGTH ANALYSIS      ")
    print("=" * 50)
    print(f"Total Sentences Analyzed : {len(token_counts)}")
    print(f"Min Token Length         : {np.min(token_counts)}")
    print(f"Max Token Length         : {np.max(token_counts)}")
    print(f"Mean Token Length        : {np.mean(token_counts):.2f}")
    print(f"Median Token Length      : {np.median(token_counts):.1f}")
    print(f"95th Percentile          : {np.percentile(token_counts, 95):.1f}")
    print(f"99th Percentile          : {np.percentile(token_counts, 99):.1f}")
    print("-" * 50)
    
    over_256 = np.sum(token_counts > 256)
    over_512 = np.sum(token_counts > 512)
    
    print(f"Sentences > 256 tokens   : {over_256} ({over_256 / len(token_counts) * 100:.2f}%)")
    print(f"Sentences > 512 tokens   : {over_512} ({over_512 / len(token_counts) * 100:.2f}%)")
    print("=" * 50)

test_pipeline = Task2DataPipeline()
analyze_dataset_token_lengths(test_pipeline, '../data/propaganda_train.tsv')

      DEBERTA SUBWORD TOKEN LENGTH ANALYSIS      
Total Sentences Analyzed : 2560
Min Token Length         : 2
Max Token Length         : 168
Mean Token Length        : 32.95
Median Token Length      : 29.0
95th Percentile          : 71.0
99th Percentile          : 101.8
--------------------------------------------------
Sentences > 256 tokens   : 0 (0.00%)
Sentences > 512 tokens   : 0 (0.00%)
